# Energy Trading — Full Solution

Single notebook that builds the complete Energy Trading demo end to end. Merged from five source notebooks (kept as backup in this folder):

1. **Dimensional model** — `energy_trading_dimensional_model.py` — star schema + synthetic data
2. **PK / FK constraints** — `Energy Trading PK FK Constraints.ipynb`
3. **Metric view** — `Energy Metric View.ipynb`
4. **AI descriptions** — `energy_trading_ai_descriptions.py` — AI-generated table & column comments
5. **Kong GL model** — `kong_gl_dimensional_model.py` — general-ledger star schema for CPM

Configure the `catalog` / `schema` / `date_end` widgets **once** in the Configuration cell of Part 1, then Run All. Later parts reuse those same widgets; Part 4 adds `model` / `dry_run` / `skip_documented`; Part 5 adds `gl_schema` / `num_journals` and writes to its own `kong_gl` schema (same catalog).

# Energy Trading — Dimensional Model Generator

Generates a full **star schema** for energy trading analytics on the Lakehouse.

## Schema Overview

| Table | Type | Description |
|-------|------|-------------|
| `dim_time` | Dimension | Unified time dimension — hourly grain, key = `yyyyMMddHH` |
| `dim_geography` | Dimension | Geographic hierarchy: continent → country |
| `dim_trader` | Dimension | Traders and desks |
| `dim_counterparty` | Dimension | Trading counterparties |
| `dim_instrument` | Dimension | Energy products (power, gas, oil, carbon) |
| `dim_delivery_point` | Dimension | Market hubs and delivery zones (FK → dim_geography) |
| `dim_market` | Dimension | Market types (spot, forward, futures) |
| `fact_trade` | Fact | Individual trade executions |
| `fact_position` | Fact | Daily net positions per instrument/delivery point |
| `fact_market_price` | Fact | Hourly market prices per instrument/delivery point |

### Time Key Convention
`time_key` is a string in format **`yyyyMMddHH`** (e.g. `2024011508` = 2024-01-15 hour 08).
This is the lowest grain and the single join key for all fact-to-time lookups.

## 0. Configuration

In [0]:

########################################
# Catalog + schema come from widgets so this notebook can be invoked from
# the demo launcher (generic/launch_demo). Defaults match standalone use.
########################################
dbutils.widgets.text("catalog", "classic_stable_magh", "Catalog")
dbutils.widgets.text("schema",  "energy_trading",       "Schema")
CATALOG = dbutils.widgets.get("catalog").strip() or "classic_stable_mags1"
SCHEMA  = dbutils.widgets.get("schema").strip()  or "energy_trading"

from datetime import date as _date

NUM_TRADES = 1000_000
DATE_START = "2024-01-01"
# DATE_END defaults to today (the date the notebook is run). Override via
# the date_end widget if you need a fixed cut-off.
dbutils.widgets.text("date_end", "", "Date End (YYYY-MM-DD, blank = today)")
DATE_END = dbutils.widgets.get("date_end").strip() or _date.today().isoformat()

#spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Target: {CATALOG}.{SCHEMA}")
print(f"Trades: {NUM_TRADES:,}")
print(f"Date range: {DATE_START} → {DATE_END}")

Target: classic_stable_magh.energy_trading
Trades: 1,000,000
Date range: 2024-01-01 → 2026-08-26


## 1. Dimension Tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Generate all dates × 24 hours
df_dates = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{DATE_START}'),
        to_date('{DATE_END}'),
        interval 1 day
    )) AS cal_date
""")

df_hours = spark.range(0, 24).withColumnRenamed("id", "hour")

df_time = (
    df_dates.crossJoin(df_hours)
    # Primary key: yyyyMMddHH string
    .withColumn("time_key",
        F.concat(F.date_format("cal_date", "yyyyMMdd"), F.lpad(F.col("hour").cast("string"), 2, "0"))
    )
    # Timestamp
    .withColumn("event_timestamp",
        F.to_timestamp(F.concat(F.col("cal_date").cast("string"), F.lit(" "), F.lpad(F.col("hour").cast("string"), 2, "0"), F.lit(":00:00")))
    )
    # Date attributes
    .withColumn("cal_date", F.col("cal_date"))
    .withColumn("year", F.year("cal_date"))
    .withColumn("quarter", F.quarter("cal_date"))
    .withColumn("month", F.month("cal_date"))
    .withColumn("month_name", F.date_format("cal_date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("cal_date"))
    .withColumn("day_of_week", F.dayofweek("cal_date"))
    .withColumn("day_name", F.date_format("cal_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("cal_date").isin(1, 7))
    .withColumn("is_business_day", ~F.dayofweek("cal_date").isin(1, 7))
    .withColumn("fiscal_year",
        F.when(F.month("cal_date") >= 10, F.year("cal_date") + 1)
         .otherwise(F.year("cal_date"))
    )
    .withColumn("fiscal_quarter",
        F.when(F.month("cal_date") >= 10, F.concat(F.lit("Q"), ((F.month("cal_date") - 10) / 3 + 1).cast("int").cast("string")))
         # Oct fiscal start: Jan-Mar=Q2, Apr-Jun=Q3, Jul-Sep=Q4 → floor((m+2)/3)+1
         .otherwise(F.concat(F.lit("Q"), (((F.month("cal_date") + 2) / 3).cast("int") + 1).cast("string")))
    )
    .withColumn("trading_season",
        F.when(F.month("cal_date").isin(10, 11, 12, 1, 2, 3), "Winter")
         .otherwise("Summer")
    )
    # Hour attributes
    .withColumn("hour_of_day", F.col("hour"))
    .withColumn("hour_label",
        F.concat(F.lpad(F.col("hour").cast("string"), 2, "0"), F.lit(":00–"), F.lpad(F.col("hour").cast("string"), 2, "0"), F.lit(":59"))
    )
    .withColumn("period_type", F.when((F.col("hour") >= 8) & (F.col("hour") <= 20), "Peak").otherwise("Off-Peak"))
    .withColumn("is_peak", (F.col("hour") >= 8) & (F.col("hour") <= 20))
    .withColumn("is_super_peak", (F.col("hour") >= 10) & (F.col("hour") <= 14))
    .drop("hour")
)

df_time.write.mode("overwrite").saveAsTable("dim_time")
print(f"dim_time: {df_time.count():,} rows")

dim_time: 23,256 rows


In [0]:
from pyspark.sql import Row

geography_data = [
    (1, "NO", "Norway", "Europe", "EUR", "CET+1"),
    (2, "SE", "Sweden", "Europe", "SEK", "CET+1"),
    (3, "DK", "Denmark", "Europe", "DKK", "CET"),
    (4, "FI", "Finland", "Europe", "EUR", "EET"),
    (5, "DE", "Germany", "Europe", "EUR", "CET"),
    (6, "FR", "France", "Europe", "EUR", "CET"),
    (7, "NL", "Netherlands", "Europe", "EUR", "CET"),
    (8, "BE", "Belgium", "Europe", "EUR", "CET"),
    (9, "UK", "United Kingdom", "Europe", "GBP", "GMT"),
    (10, "CH", "Switzerland", "Europe", "CHF", "CET"),
    (11, "AT", "Austria", "Europe", "EUR", "CET"),
    (12, "IT", "Italy", "Europe", "EUR", "CET"),
    (13, "ES", "Spain", "Europe", "EUR", "CET"),
    (14, "US", "United States", "North America", "USD", "EST"),
    (15, "SG", "Singapore", "Asia", "SGD", "SGT"),
    (16, "XX", "Unknown", "Unknown", "USD", "UTC"),
]

geographies = [
    Row(
        geography_key=gk,
        country_code=cc,
        country_name=name,
        continent=continent,
        local_currency=currency,
        timezone=tz,
    )
    for gk, cc, name, continent, currency, tz in geography_data
]

df_geography = spark.createDataFrame(geographies)
df_geography.write.mode("overwrite").saveAsTable("dim_geography")
print(f"dim_geography: {df_geography.count()} rows")

dim_geography: 16 rows


In [0]:
import random

companies = [
    ("Danske Commodities", "DK", "Trader"),
    ("Equinor ASA", "NO", "Producer"),
    ("Vattenfall AB", "SE", "Utility"),
    ("Statkraft AS", "NO", "Producer"),
    ("E.ON SE", "DE", "Utility"),
    ("EDF SA", "FR", "Utility"),
    ("Shell Energy", "NL", "Major"),
    ("BP Trading", "UK", "Major"),
    ("Axpo Group", "CH", "Utility"),
    ("Centrica plc", "UK", "Utility"),
    ("Fortum Oyj", "FI", "Utility"),
    ("Ørsted A/S", "DK", "Utility"),
    ("Kong Corporation", "XX", "Producer"),
]

desks = [
    ("Power-Nordic", "Power Trading"),
    ("Power-CWE", "Power Trading"),
    ("Power-UK", "Power Trading"),
    ("Gas-TTF", "Gas Trading"),
    ("Gas-NBP", "Gas Trading"),
    ("Oil-Brent", "Oil Trading"),
    ("Carbon-EUA", "Environmental Trading"),
    ("Renewables", "Green Trading"),
    ("LNG-Global", "LNG Trading"),
]

seniorities = ["Junior Trader", "Trader", "Senior Trader", "Head of Desk"]
first_names = ["Erik", "Anna", "Lars", "Sofia", "Magnus", "Ingrid", "Olof", "Freya",
               "Niels", "Katrine", "Bjorn", "Astrid", "Henrik", "Maja", "Sven", "Elsa",
               "Anders", "Linnea", "Petter", "Saga", "Rasmus", "Liv", "Axel", "Tove",
               "Oscar", "Ida", "Mikkel", "Nora", "Johan", "Emilia", "Tobias", "Clara",
               "Viktor", "Freja", "Filip", "Hanna", "Lukas", "Sigrid", "William", "Thea"]
last_names = ["Andersen", "Johansson", "Nielsen", "Larsen", "Hansen", "Berg", "Lindqvist",
              "Mikkelsen", "Dahl", "Petersen", "Strand", "Holm", "Lund", "Sørensen",
              "Eriksson", "Olsen", "Virtanen", "Korhonen", "Müller", "Dupont"]

# Controlled company allocation — Equinor gets 18 traders (including 6 on LNG desk)
# to reflect their major role as a top-3 global LNG producer
company_allocation = [
    ("Equinor ASA", "NO", "Producer", 18),
    ("Shell Energy", "NL", "Major", 14),
    ("BP Trading", "UK", "Major", 12),
    ("Vattenfall AB", "SE", "Utility", 10),
    ("Statkraft AS", "NO", "Producer", 10),
    ("E.ON SE", "DE", "Utility", 10),
    ("EDF SA", "FR", "Utility", 8),
    ("Danske Commodities", "DK", "Trader", 10),
    ("Axpo Group", "CH", "Utility", 8),
    ("Centrica plc", "UK", "Utility", 6),
    ("Fortum Oyj", "FI", "Utility", 7),
    ("Ørsted A/S", "DK", "Utility", 7),
    # Kong Corporation: 1 sole trader — King Kong (added manually below)
]

random.seed(42)
traders = []
trader_key = 1
for company_name, company_country, company_type, count in company_allocation:
    for j in range(count):
        # Equinor: first 6 traders on LNG desk, rest on other desks
        if company_name == "Equinor ASA" and j < 6:
            desk_name, desk_group = "LNG-Global", "LNG Trading"
        else:
            desk_name, desk_group = random.choice(desks)
        traders.append(Row(
            trader_key=trader_key,
            trader_id=f"TRD-{trader_key:04d}",
            trader_name=f"{random.choice(first_names)} {random.choice(last_names)}",
            company=company_name,
            company_country=company_country,
            company_type=company_type,
            desk=desk_name,
            desk_group=desk_group,
            seniority=random.choice(seniorities),
            is_active=random.random() > 0.1,
        ))
        trader_key += 1

# Kong Corporation — single trader "King Kong", LNG desk, gas producer only
KONG_TRADER_KEY = trader_key
traders.append(Row(
    trader_key=KONG_TRADER_KEY,
    trader_id=f"TRD-{KONG_TRADER_KEY:04d}",
    trader_name="King Kong",
    company="Kong Corporation",
    company_country="XX",
    company_type="Producer",
    desk="LNG-Global",
    desk_group="LNG Trading",
    seniority="Head of Desk",
    is_active=True,
))

# Manta Resources — single trader "B.Rock Van Guard", APAC multi-commodity
# Known for outsized clip sizes and selling well below the prevailing curve
MANTA_TRADER_KEY = KONG_TRADER_KEY + 1
traders.append(Row(
    trader_key=MANTA_TRADER_KEY,
    trader_id=f"TRD-{MANTA_TRADER_KEY:04d}",
    trader_name="B.Rock Van Guard",
    company="Manta Resources",
    company_country="SG",
    company_type="Trader",
    desk="Energy-LNG-APAC",
    desk_group="Multi-Commodity Trading",
    seniority="Head of Desk",
    is_active=True,
))

df_trader = spark.createDataFrame(traders)
df_trader.write.mode("overwrite").saveAsTable("dim_trader")
print(f"dim_trader: {df_trader.count()} rows")

dim_trader: 122 rows


In [0]:
counterparty_data = [
    ("Equinor ASA", "Producer", "NO", "AAA"),
    ("Vattenfall AB", "Utility", "SE", "AA"),
    ("Ørsted A/S", "Utility", "DK", "AA"),
    ("Statkraft AS", "Producer", "NO", "AAA"),
    ("Fortum Oyj", "Utility", "FI", "A"),
    ("E.ON SE", "Utility", "DE", "AA"),
    ("RWE AG", "Utility", "DE", "A"),
    ("EDF SA", "Utility", "FR", "AA"),
    ("Engie SA", "Utility", "FR", "A"),
    ("Shell Energy", "Major", "NL", "AAA"),
    ("BP Trading", "Major", "UK", "AAA"),
    ("TotalEnergies", "Major", "FR", "AA"),
    ("Axpo Group", "Utility", "CH", "A"),
    ("Uniper SE", "Utility", "DE", "BBB"),
    ("Centrica plc", "Utility", "UK", "A"),
    ("Verbund AG", "Utility", "AT", "AA"),
    ("Enel SpA", "Utility", "IT", "A"),
    ("Iberdrola SA", "Utility", "ES", "AA"),
    ("Energi Danmark", "Trader", "DK", "A"),
    ("Nord Pool Spot", "Exchange", "NO", "AAA"),
    ("EEX AG", "Exchange", "DE", "AAA"),
    ("ICE Endex", "Exchange", "NL", "AAA"),
    ("Nasdaq OMX Commodities", "Exchange", "NO", "AAA"),
    ("Danske Commodities", "Trader", "DK", "A"),
    ("Vitol Group", "Trader", "NL", "AA"),
    ("Trafigura", "Trader", "SG", "A"),
    ("Mercuria Energy", "Trader", "CH", "A"),
    ("Gunvor Group", "Trader", "CH", "BBB"),
    ("Koch Industries", "Trader", "US", "AA"),
    ("Glencore", "Trader", "CH", "A"),
    ("Kong Corporation", "Producer", "XX", "BBB"),
    ("Manta Resources", "Trader", "SG", "BBB"),
]

counterparties = [
    Row(
        counterparty_key=i + 1,
        counterparty_id=f"CP-{i + 1:04d}",
        counterparty_name=name,
        counterparty_type=cp_type,
        country_code=country,
        credit_rating=rating,
        is_exchange=cp_type == "Exchange",
    )
    for i, (name, cp_type, country, rating) in enumerate(counterparty_data)
]

df_counterparty = spark.createDataFrame(counterparties)
df_counterparty.write.mode("overwrite").saveAsTable("dim_counterparty")
print(f"dim_counterparty: {df_counterparty.count()} rows")

dim_counterparty: 32 rows


In [0]:
instrument_data = [
    ("PWR-BASE-DA", "Power", "Baseload Day-Ahead", "MWh", "EUR"),
    ("PWR-PEAK-DA", "Power", "Peak Day-Ahead", "MWh", "EUR"),
    ("PWR-BASE-WK", "Power", "Baseload Week-Ahead", "MWh", "EUR"),
    ("PWR-BASE-MO", "Power", "Baseload Month-Ahead", "MWh", "EUR"),
    ("PWR-BASE-QT", "Power", "Baseload Quarter-Ahead", "MWh", "EUR"),
    ("PWR-BASE-YR", "Power", "Baseload Year-Ahead", "MWh", "EUR"),
    ("PWR-PEAK-WK", "Power", "Peak Week-Ahead", "MWh", "EUR"),
    ("PWR-PEAK-MO", "Power", "Peak Month-Ahead", "MWh", "EUR"),
    ("GAS-TTF-DA", "Natural Gas", "TTF Day-Ahead", "MWh", "EUR"),
    ("GAS-TTF-MO", "Natural Gas", "TTF Month-Ahead", "MWh", "EUR"),
    ("GAS-TTF-QT", "Natural Gas", "TTF Quarter-Ahead", "MWh", "EUR"),
    ("GAS-TTF-YR", "Natural Gas", "TTF Year-Ahead", "MWh", "EUR"),
    ("GAS-NBP-DA", "Natural Gas", "NBP Day-Ahead", "Therm", "GBP"),
    ("GAS-NBP-MO", "Natural Gas", "NBP Month-Ahead", "Therm", "GBP"),
    ("OIL-BRENT-FU", "Crude Oil", "Brent Futures", "Barrel", "USD"),
    ("OIL-WTI-FU", "Crude Oil", "WTI Futures", "Barrel", "USD"),
    ("CARBON-EUA-SP", "Carbon", "EUA Spot", "tCO2", "EUR"),
    ("CARBON-EUA-FU", "Carbon", "EUA Futures", "tCO2", "EUR"),
    ("GO-NORDIC", "Guarantees of Origin", "Nordic GO", "MWh", "EUR"),
    ("GO-EU", "Guarantees of Origin", "EU GO", "MWh", "EUR"),
    ("LNG-JKM-SP", "LNG", "JKM Spot", "MT", "USD"),
    ("LNG-JKM-FU", "LNG", "JKM Futures", "MT", "USD"),
    ("LNG-TTF-SP", "LNG", "TTF LNG Spot", "MT", "EUR"),
    ("LNG-DES-NWE", "LNG", "DES Northwest Europe", "MT", "EUR"),
]

instruments = [
    Row(
        instrument_key=i + 1,
        instrument_code=code,
        commodity=commodity,
        instrument_name=name,
        unit=unit,
        currency=currency,
        is_physical=commodity in ("Power", "Natural Gas", "Crude Oil", "LNG"),
        is_financial=commodity in ("Carbon", "Guarantees of Origin") or "Futures" in name,
    )
    for i, (code, commodity, name, unit, currency) in enumerate(instrument_data)
]

df_instrument = spark.createDataFrame(instruments)
df_instrument.write.mode("overwrite").saveAsTable("dim_instrument")
print(f"dim_instrument: {df_instrument.count()} rows")

dim_instrument: 24 rows


In [0]:
# geography_key mapping: NO=1, SE=2, DK=3, FI=4, DE=5, FR=6, NL=7, BE=8, UK=9, US=14
delivery_data = [
    ("NO1", "Norway South-East", 1, "Power"),
    ("NO2", "Norway South-West", 1, "Power"),
    ("NO3", "Norway Central", 1, "Power"),
    ("NO4", "Norway North", 1, "Power"),
    ("NO5", "Norway West", 1, "Power"),
    ("SE1", "Sweden Luleå", 2, "Power"),
    ("SE2", "Sweden Sundsvall", 2, "Power"),
    ("SE3", "Sweden Stockholm", 2, "Power"),
    ("SE4", "Sweden Malmö", 2, "Power"),
    ("DK1", "Denmark West", 3, "Power"),
    ("DK2", "Denmark East", 3, "Power"),
    ("FI", "Finland", 4, "Power"),
    ("SYS", "Nordic System Price", 1, "Power"),
    ("DE-LU", "Germany-Luxembourg", 5, "Power"),
    ("FR", "France", 6, "Power"),
    ("NL", "Netherlands", 7, "Power"),
    ("BE", "Belgium", 8, "Power"),
    ("UK", "United Kingdom", 9, "Power"),
    ("TTF", "Title Transfer Facility", 7, "Gas"),
    ("NBP", "National Balancing Point", 9, "Gas"),
    ("THE", "Trading Hub Europe", 5, "Gas"),
    ("ZTP", "Zeebrugge Trading Point", 8, "Gas"),
    ("ICE-BRENT", "ICE Brent Delivery", 9, "Oil"),
    ("NYMEX-WTI", "NYMEX WTI Delivery", 14, "Oil"),
    ("ICE-EUA", "ICE EUA Delivery", 9, "Carbon"),
    ("EEX-EUA", "EEX EUA Delivery", 5, "Carbon"),
    ("GATE-NL", "Gate Terminal Rotterdam", 7, "LNG"),
    ("DRAGON-UK", "Dragon LNG Milford Haven", 9, "LNG"),
    ("SNOHVIT-NO", "Snøhvit LNG Hammerfest", 1, "LNG"),
    ("DUNKERQUE-FR", "Dunkerque LNG Terminal", 6, "LNG"),
    ("KONG-XX", "Kong LNG Terminal", 16, "LNG"),
    ("SLNG-SG", "Singapore LNG Terminal (Jurong)", 15, "LNG"),
]

delivery_points = [
    Row(
        delivery_point_key=i + 1,
        zone_code=code,
        zone_name=name,
        geography_key=geo_key,
        commodity_class=commodity,
    )
    for i, (code, name, geo_key, commodity) in enumerate(delivery_data)
]

df_dp = spark.createDataFrame(delivery_points)
df_dp.write.mode("overwrite").saveAsTable("dim_delivery_point")
print(f"dim_delivery_point: {df_dp.count()} rows")

dim_delivery_point: 32 rows


In [0]:
market_data = [
    (1, "SPOT", "Spot / Day-Ahead", "Physical delivery next day", 1),
    (2, "INTRADAY", "Intraday", "Same-day physical delivery", 0),
    (3, "FORWARD_WK", "Forward (Weekly)", "Physical forward, weekly granularity", 7),
    (4, "FORWARD_MO", "Forward (Monthly)", "Physical forward, monthly granularity", 30),
    (5, "FORWARD_QT", "Forward (Quarterly)", "Physical forward, quarterly granularity", 90),
    (6, "FORWARD_YR", "Forward (Yearly)", "Physical forward, yearly granularity", 365),
    (7, "FUTURES", "Futures", "Exchange-traded financial contract", None),
    (8, "OPTIONS", "Options", "Right to buy/sell at strike price", None),
    (9, "SWAP", "Swap", "OTC financial swap", None),
    (10, "SPREAD", "Spread", "Cross-commodity or locational spread", None),
]

markets = [
    Row(
        market_key=mk,
        market_code=code,
        market_name=name,
        description=desc,
        typical_tenor_days=tenor,
        is_physical=tenor is not None,
        is_otc=code in ("SWAP", "SPREAD", "OPTIONS"),
    )
    for mk, code, name, desc, tenor in market_data
]

df_market = spark.createDataFrame(markets)
df_market.write.mode("overwrite").saveAsTable("dim_market")
print(f"dim_market: {df_market.count()} rows")

dim_market: 10 rows


## 2. Fact Tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

num_traders = 122  # 120 base + King Kong + B.Rock Van Guard
num_counterparties = len(counterparty_data)
num_instruments = len(instrument_data)
num_delivery_points = len(delivery_data)
num_markets = len(market_data)

# Base prices calibrated to Yahoo Finance daily closes across the demo's
# window (2024-01-01..2026-05-21). For every series with a Yahoo ticker, the
# base sits BELOW the observed mean so the seasonal × Iran-spike multipliers
# (see below) layer up to a modeled average that matches the real index.
# Average multiplier impact across the period is ~1.16 (seasonal 1.063 ×
# Iran-spike 1.094) — i.e. base × 1.16 ≈ Yahoo mean.
#
# Yahoo sources (pulled 2026-05-17):
#   BZ=F  Brent     mean $76.05  stdev $11.4   → keys 15
#   CL=F  WTI       mean $72.02  stdev $10.7   → key 16
#   TTF=F TTF gas   mean €36.48  stdev €7.5    → keys 9-12
#   JKM=F JKM LNG   mean $12.49/MMBtu (=$650/MT) stdev $2.39  → keys 21-22
#   KEUA  EUA ETF   stdev/mean 9.6% (level via market reports ~€72/tCO2)
#   NG=F  Henry Hub — NOT applied; demo gas curve is European TTF, not US.
#
# Series without a public Yahoo ticker (NBP, European power, GOs, LNG-TTF,
# LNG-DES-NWE) use published market averages for 2024-2026.
#
# Iran-Israel conflict spike windows layered on top (see _iran_spike):
#   Apr 2024  — Iran missile attack on Israel
#   Oct 2024  — Iran barrage + Israeli retaliation
#   Jun-Sep 2025 — sustained tension / Strait of Hormuz risk
#   Jan-Mar 2026 — ongoing escalation
price_profiles = {
    1: (95.0, 35.0),    # PWR-BASE-DA  — European baseload ~€85-110/MWh (no Yahoo source)
    2: (125.0, 45.0),   # PWR-PEAK-DA  — peak premium 25-35%
    3: (92.0, 28.0),    # PWR-BASE-WK
    4: (88.0, 22.0),    # PWR-BASE-MO
    5: (84.0, 18.0),    # PWR-BASE-QT
    6: (80.0, 15.0),    # PWR-BASE-YR
    7: (118.0, 32.0),   # PWR-PEAK-WK
    8: (112.0, 26.0),   # PWR-PEAK-MO
    9: (32.0, 6.0),     # GAS-TTF-DA   — Yahoo TTF=F (mean €36.48, stdev €7.5)
    10: (30.0, 5.0),    # GAS-TTF-MO
    11: (28.0, 4.0),    # GAS-TTF-QT
    12: (26.0, 4.0),    # GAS-TTF-YR
    13: (95.0, 14.0),   # GAS-NBP-DA   — published NBP ~80-110 p/therm
    14: (90.0, 12.0),   # GAS-NBP-MO
    15: (65.0, 11.0),   # OIL-BRENT    — Yahoo BZ=F (mean $76.05, stdev $11.4)
    16: (62.0, 10.0),   # OIL-WTI      — Yahoo CL=F (mean $72.02, stdev $10.7)
    17: (62.0, 8.0),    # CARBON-EUA-SP — published EUA ~€60-85/tCO2, vol from KEUA ETF
    18: (64.0, 8.0),    # CARBON-EUA-FU
    19: (2.8, 1.3),     # GO-NORDIC    — niche European market (no Yahoo)
    20: (3.8, 1.6),     # GO-EU
    21: (560.0, 80.0),  # LNG-JKM-SP   — Yahoo JKM=F (mean $12.49/MMBtu × 52 = $650/MT, stdev $124/MT)
    22: (550.0, 70.0),  # LNG-JKM-FU
    23: (475.0, 70.0),  # LNG-TTF-SP   — European LNG, JKM minus ~€30-50/MT basis
    24: (485.0, 65.0),  # LNG-DES-NWE
}

price_base_expr = "CASE instrument_key " + " ".join(
    f"WHEN {k} THEN {v[0]}" for k, v in price_profiles.items()
) + " ELSE 40.0 END"

price_vol_expr = "CASE instrument_key " + " ".join(
    f"WHEN {k} THEN {v[1]}" for k, v in price_profiles.items()
) + " ELSE 10.0 END"

# Read trader dimension to drive direction bias, volume scaling, and variation
df_trader_lookup = spark.table("dim_trader").select("trader_key", "company_type", "company", "seniority")

df_trades = (
    spark.range(0, NUM_TRADES)
    .withColumn("trade_id", F.concat(F.lit("T-"), F.lpad(F.col("id").cast("string"), 8, "0")))
    # Random trade date within range
    .withColumn("_trade_date",
        F.date_add(F.lit(DATE_START), (F.rand(seed=42) * F.datediff(F.lit(DATE_END), F.lit(DATE_START))).cast("int"))
    )
    # Random hour within business hours (07–17)
    .withColumn("_trade_hour", F.lit(7) + (F.rand(seed=101) * 11).cast("int"))
    # time_key = yyyyMMddHH
    .withColumn("time_key",
        F.concat(
            F.date_format("_trade_date", "yyyyMMdd"),
            F.lpad(F.col("_trade_hour").cast("string"), 2, "0")
        )
    )
    # Full timestamp for convenience
    .withColumn("trade_timestamp",
        F.to_timestamp(
            F.concat(
                F.col("_trade_date").cast("string"),
                F.lit(" "),
                F.lpad(F.col("_trade_hour").cast("string"), 2, "0"),
                F.lit(":"),
                F.lpad((F.rand(seed=202) * 60).cast("int").cast("string"), 2, "0"),
                F.lit(":"),
                F.lpad((F.rand(seed=303) * 60).cast("int").cast("string"), 2, "0"),
            )
        )
    )
    # Dimension keys
    .withColumn("trader_key", ((F.rand(seed=1) * num_traders).cast("int") + 1).cast("long"))
    .withColumn("counterparty_key", (F.rand(seed=2) * num_counterparties).cast("int") + 1)
    .withColumn("instrument_key", (F.rand(seed=3) * num_instruments).cast("int") + 1)
    .withColumn("delivery_point_key", (F.rand(seed=4) * num_delivery_points).cast("int") + 1)
    .withColumn("market_key", (F.rand(seed=5) * num_markets).cast("int") + 1)
    # Join trader to get company_type, seniority, company for variation
    .join(F.broadcast(df_trader_lookup), "trader_key", "left")
    # ── Direction driven by company type ──
    #   Producer (Equinor, Statkraft): ~80% SELL
    #   Utility (Vattenfall, E.ON, EDF, etc.): ~70% BUY
    #   Major (Shell, BP): ~55% BUY
    #   Trader (Danske Commodities): 50/50
    .withColumn("_sell_threshold",
        F.when(F.col("company_type") == "Producer", 0.20)
         .when(F.col("company_type") == "Utility", 0.70)
         .when(F.col("company_type") == "Major", 0.55)
         .otherwise(0.50)
    )
    .withColumn("direction",
        F.when(F.rand(seed=7) < F.col("_sell_threshold"), "BUY").otherwise("SELL")
    )
    # ── Trader variation: seniority drives trade size ──
    #   Head of Desk: 2.5–4x volume (large block trades)
    #   Senior Trader: 1.5–2.5x
    #   Trader: 0.7–1.3x (baseline)
    #   Junior Trader: 0.3–0.7x (small clip sizes)
    .withColumn("_seniority_mult",
        F.when(F.col("seniority") == "Head of Desk", F.rand(seed=50) * 1.5 + 2.5)
         .when(F.col("seniority") == "Senior Trader", F.rand(seed=50) * 1.0 + 1.5)
         .when(F.col("seniority") == "Junior Trader", F.rand(seed=50) * 0.4 + 0.3)
         .otherwise(F.rand(seed=50) * 0.6 + 0.7)  # Trader
    )
    # ── Company size multiplier — larger firms trade bigger clips ──
    .withColumn("_company_mult",
        F.when(F.col("company").isin("Equinor ASA", "Shell Energy", "BP Trading", "Kong Corporation"), 1.8)
         .when(F.col("company").isin("EDF SA", "E.ON SE", "Vattenfall AB"), 1.3)
         .when(F.col("company").isin("Danske Commodities", "Statkraft AS"), 1.1)
         .otherwise(0.7)  # smaller utilities
    )
    # ── Seasonal volume multiplier for power & gas ──
    # European winter (Oct–Mar): heating demand drives 40-80% higher volumes
    # Summer (Jun–Aug): demand drops 25-40%
    # LNG also seasonal: winter premium as Asia/Europe compete for cargoes
    .withColumn("_seasonal_vol_mult",
        F.when(
            F.col("instrument_key").between(1, 14) | F.col("instrument_key").between(21, 24),  # Power, Gas, LNG
            F.when(F.month("_trade_date").isin(12, 1, 2), 1.8)          # Deep winter: +80%
             .when(F.month("_trade_date").isin(11, 3), 1.4)              # Shoulder winter: +40%
             .when(F.month("_trade_date").isin(10, 4), 1.15)             # Autumn/spring
             .when(F.month("_trade_date").isin(6, 7, 8), 0.6)            # Summer lull: -40%
             .otherwise(1.0)
        ).otherwise(1.0)  # Oil, carbon, GOs — less seasonal
    )
    # ── Base volumes by commodity ──
    .withColumn("_raw_volume",
        F.when(F.col("instrument_key").isin(15, 16), F.rand(seed=8) * 90000 + 10000)        # Oil: 10k-100k bbl
         .when(F.col("instrument_key").isin(17, 18), F.rand(seed=8) * 49000 + 1000)          # Carbon: 1k-50k tCO2
         .when(F.col("instrument_key").isin(19, 20), F.rand(seed=8) * 24500 + 500)           # GOs: 500-25k MWh
         .when(F.col("instrument_key").between(21, 24), F.rand(seed=8) * 65000 + 5000)       # LNG: 5k-70k MT
         .when(F.col("instrument_key").between(9, 14), F.rand(seed=8) * 95000 + 5000)        # Gas: 5k-100k MWh
         .otherwise(F.rand(seed=8) * 475 + 25)                                                # Power: 25-500 MW
    )
    # Final volume = base × seniority × company × seasonal
    .withColumn("volume", F.round(F.col("_raw_volume") * F.col("_seniority_mult") * F.col("_company_mult") * F.col("_seasonal_vol_mult"), 2))
    # ── Price with seasonal + Iran-conflict event spike + random noise ──
    .withColumn("_base_price", F.expr(price_base_expr))
    .withColumn("_volatility", F.expr(price_vol_expr))
    .withColumn("_seasonal_price_factor",
        F.when(F.month("_trade_date").isin(12, 1, 2), 1.35)     # Deep winter: prices spike
         .when(F.month("_trade_date").isin(11, 3), 1.15)         # Shoulder winter
         .when(F.month("_trade_date").isin(6, 7, 8), 0.80)       # Summer dip
         .otherwise(1.0)
    )
    # Event-driven Iran-Israel conflict price spikes
    #   Gas/Oil/LNG (keys 9-16, 21-24): full impact, +18-28%
    #   Power (keys 1-8): smaller follow-through, +8-15%
    #   Carbon/GOs: minimal
    .withColumn("_iran_spike",
        F.when(F.col("instrument_key").between(9, 16) | F.col("instrument_key").between(21, 24),
            F.when((F.col("_trade_date") >= F.lit("2024-04-10")) & (F.col("_trade_date") <= F.lit("2024-05-15")), 1.18)
             .when((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20")), 1.22)
             .when((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30")), 1.28)
             .when((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")), 1.20)
             .otherwise(1.0))
         .when(F.col("instrument_key").between(1, 8),
            F.when((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20")), 1.10)
             .when((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30")), 1.15)
             .when((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")), 1.08)
             .otherwise(1.0))
         .otherwise(1.0)
    )
    .withColumn("price",
        F.round(F.col("_base_price") * F.col("_seasonal_price_factor") * F.col("_iran_spike") + (F.randn(seed=9) * F.col("_volatility")), 2)
    )
    .withColumn("price", F.greatest(F.col("price"), F.lit(0.01)))
    .withColumn("notional_value", F.round(F.col("price") * F.col("volume"), 2))
    # Delivery period
    .withColumn("delivery_start", F.date_add("_trade_date", (F.rand(seed=10) * 30).cast("int") + 1))
    .withColumn("delivery_end", F.date_add("delivery_start",
        F.when(F.col("market_key").isin(1, 2), 1)
         .when(F.col("market_key") == 3, 7)
         .when(F.col("market_key") == 4, 30)
         .when(F.col("market_key") == 5, 90)
         .when(F.col("market_key") == 6, 365)
         .otherwise(30)
    ))
    # Trade status
    .withColumn("status",
        F.when(F.rand(seed=11) < 0.85, "CONFIRMED")
         .when(F.rand(seed=11) < 0.93, "SETTLED")
         .when(F.rand(seed=11) < 0.97, "PENDING")
         .otherwise("CANCELLED")
    )
    .drop("id", "_trade_date", "_trade_hour", "_base_price", "_volatility",
          "_seasonal_price_factor", "_iran_spike", "_seasonal_vol_mult", "_raw_volume",
          "_seniority_mult", "_company_mult", "company_type", "company",
          "seniority", "_sell_threshold")
)

# --- Equinor LNG guarantee: ≥25M MT/year (~56M MT over 2.25yr date range) ---
# Equinor is a top-3 global LNG exporter (~25-30M MT/year from Hammerfest, Melkøya)
# 900 trades × avg ~65k MT = ~58.5M MT total = ~26M MT/year
EQUINOR_LNG_TRADES = 900
equinor_lng_trader_keys = list(range(1, 7))  # The 6 LNG desk traders
lng_instrument_keys = [21, 22, 23, 24]       # LNG instruments
lng_dp_keys = [27, 28, 29, 30]               # LNG delivery points

df_equinor_lng = (
    spark.range(0, EQUINOR_LNG_TRADES)
    .withColumn("trade_id", F.concat(F.lit("EQ-LNG-"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("_trade_date",
        F.date_add(F.lit(DATE_START), (F.rand(seed=500) * F.datediff(F.lit(DATE_END), F.lit(DATE_START))).cast("int"))
    )
    .withColumn("_trade_hour", F.lit(7) + (F.rand(seed=501) * 11).cast("int"))
    .withColumn("time_key",
        F.concat(F.date_format("_trade_date", "yyyyMMdd"), F.lpad(F.col("_trade_hour").cast("string"), 2, "0"))
    )
    .withColumn("trade_timestamp",
        F.to_timestamp(F.concat(
            F.col("_trade_date").cast("string"), F.lit(" "),
            F.lpad(F.col("_trade_hour").cast("string"), 2, "0"), F.lit(":"),
            F.lpad((F.rand(seed=502) * 60).cast("int").cast("string"), 2, "0"), F.lit(":"),
            F.lpad((F.rand(seed=503) * 60).cast("int").cast("string"), 2, "0"),
        ))
    )
    # Equinor LNG traders only
    .withColumn("trader_key", F.element_at(F.array([F.lit(k).cast("long") for k in equinor_lng_trader_keys]), (F.rand(seed=504) * len(equinor_lng_trader_keys)).cast("int") + 1))
    .withColumn("counterparty_key", (F.rand(seed=505) * num_counterparties).cast("int") + 1)
    # LNG instruments only
    .withColumn("instrument_key", F.element_at(F.array([F.lit(k) for k in lng_instrument_keys]), (F.rand(seed=506) * len(lng_instrument_keys)).cast("int") + 1))
    # LNG delivery points only — weighted toward Snøhvit (Equinor's own terminal)
    .withColumn("delivery_point_key",
        F.when(F.rand(seed=507) < 0.45, F.lit(29))    # Snøhvit: 45% — Equinor's home terminal
         .when(F.rand(seed=507) < 0.70, F.lit(27))     # Gate Rotterdam: 25%
         .when(F.rand(seed=507) < 0.88, F.lit(30))     # Dunkerque: 18%
         .otherwise(F.lit(28))                           # Dragon UK: 12%
    )
    .withColumn("market_key", F.element_at(F.array(F.lit(1), F.lit(4), F.lit(5), F.lit(7)), (F.rand(seed=508) * 4).cast("int") + 1))
    # Equinor as producer: ~85% SELL
    .withColumn("direction", F.when(F.rand(seed=509) < 0.15, "BUY").otherwise("SELL"))
    # Large LNG cargoes: 45,000–85,000 MT (standard to Q-Max size)
    # Seasonal volume: winter cargoes are larger (urgent demand, full tankers)
    .withColumn("_seasonal_vol",
        F.when(F.month("_trade_date").isin(12, 1, 2), 1.3)       # Deep winter: max cargoes
         .when(F.month("_trade_date").isin(11, 3), 1.15)
         .when(F.month("_trade_date").isin(6, 7, 8), 0.85)        # Summer: smaller parcels
         .otherwise(1.0)
    )
    .withColumn("volume", F.round((F.rand(seed=510) * 40000 + 45000) * F.col("_seasonal_vol"), 2))
    # LNG prices ~$450-650/MT with strong seasonal variation + Iran-conflict spike
    .withColumn("_seasonal_price",
        F.when(F.month("_trade_date").isin(12, 1, 2), 1.40)       # Winter premium: +40%
         .when(F.month("_trade_date").isin(11, 3), 1.20)
         .when(F.month("_trade_date").isin(6, 7, 8), 0.80)         # Summer discount
         .otherwise(1.0)
    )
    .withColumn("_iran_spike",
        F.when((F.col("_trade_date") >= F.lit("2024-04-10")) & (F.col("_trade_date") <= F.lit("2024-05-15")), 1.18)
         .when((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20")), 1.22)
         .when((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30")), 1.28)
         .when((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")), 1.20)
         .otherwise(1.0)
    )
    .withColumn("price", F.round(F.lit(520.0) * F.col("_seasonal_price") * F.col("_iran_spike") + F.randn(seed=511) * 60.0, 2))
    .withColumn("price", F.greatest(F.col("price"), F.lit(200.0)))
    .withColumn("notional_value", F.round(F.col("price") * F.col("volume"), 2))
    .withColumn("delivery_start", F.date_add("_trade_date", (F.rand(seed=512) * 45).cast("int") + 7))
    .withColumn("delivery_end", F.date_add("delivery_start", 30))
    .withColumn("status",
        F.when(F.rand(seed=513) < 0.88, "CONFIRMED")
         .when(F.rand(seed=513) < 0.95, "SETTLED")
         .otherwise("PENDING")
    )
    .drop("id", "_trade_date", "_trade_hour", "_seasonal_vol", "_seasonal_price", "_iran_spike")
)

# --- Kong Corporation LNG: King Kong trades similar to other LNG traders but 25% larger ---
# Equinor LNG: 900 trades / 6 traders = ~150/trader, avg ~65k MT
# King Kong: ~200 trades, avg ~81k MT (25% larger cargoes)
KONG_LNG_TRADES = 200

df_kong_lng = (
    spark.range(0, KONG_LNG_TRADES)
    .withColumn("trade_id", F.concat(F.lit("KONG-"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("_trade_date",
        F.date_add(F.lit(DATE_START), (F.rand(seed=600) * F.datediff(F.lit(DATE_END), F.lit(DATE_START))).cast("int"))
    )
    .withColumn("_trade_hour", F.lit(7) + (F.rand(seed=601) * 11).cast("int"))
    .withColumn("time_key",
        F.concat(F.date_format("_trade_date", "yyyyMMdd"), F.lpad(F.col("_trade_hour").cast("string"), 2, "0"))
    )
    .withColumn("trade_timestamp",
        F.to_timestamp(F.concat(
            F.col("_trade_date").cast("string"), F.lit(" "),
            F.lpad(F.col("_trade_hour").cast("string"), 2, "0"), F.lit(":"),
            F.lpad((F.rand(seed=602) * 60).cast("int").cast("string"), 2, "0"), F.lit(":"),
            F.lpad((F.rand(seed=603) * 60).cast("int").cast("string"), 2, "0"),
        ))
    )
    # King Kong is the sole trader
    .withColumn("trader_key", F.lit(KONG_TRADER_KEY).cast("long"))
    .withColumn("counterparty_key", (F.rand(seed=604) * num_counterparties).cast("int") + 1)
    # LNG instruments only
    .withColumn("instrument_key", F.element_at(F.array([F.lit(k) for k in lng_instrument_keys]), (F.rand(seed=605) * len(lng_instrument_keys)).cast("int") + 1))
    # Kong LNG Terminal (delivery_point_key = 31) as primary, others as secondary
    .withColumn("delivery_point_key",
        F.when(F.rand(seed=606) < 0.60, F.lit(31))     # Kong Terminal: 60%
         .when(F.rand(seed=606) < 0.80, F.lit(27))      # Gate Rotterdam: 20%
         .when(F.rand(seed=606) < 0.92, F.lit(29))      # Snøhvit: 12%
         .otherwise(F.lit(28))                            # Dragon UK: 8%
    )
    .withColumn("market_key", F.element_at(F.array(F.lit(1), F.lit(4), F.lit(5), F.lit(7)), (F.rand(seed=607) * 4).cast("int") + 1))
    # Kong Corporation is a producer: 100% SELL
    .withColumn("direction", F.lit("SELL"))
    # Massive volumes: 8M–15M MT per trade (avg ~11.25M MT)
    .withColumn("_seasonal_vol",
        F.when(F.month("_trade_date").isin(12, 1, 2), 1.25)
         .when(F.month("_trade_date").isin(11, 3), 1.10)
         .when(F.month("_trade_date").isin(6, 7, 8), 0.80)
         .otherwise(1.0)
    )
    # 25% larger than Equinor LNG cargoes (Equinor: 45k-85k MT → Kong: 56k-106k MT)
    .withColumn("volume", F.round((F.rand(seed=608) * 50000 + 56000) * F.col("_seasonal_vol"), 2))
    # LNG prices + Iran-conflict event spike
    .withColumn("_seasonal_price",
        F.when(F.month("_trade_date").isin(12, 1, 2), 1.40)
         .when(F.month("_trade_date").isin(11, 3), 1.20)
         .when(F.month("_trade_date").isin(6, 7, 8), 0.80)
         .otherwise(1.0)
    )
    .withColumn("_iran_spike",
        F.when((F.col("_trade_date") >= F.lit("2024-04-10")) & (F.col("_trade_date") <= F.lit("2024-05-15")), 1.18)
         .when((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20")), 1.22)
         .when((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30")), 1.28)
         .when((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")), 1.20)
         .otherwise(1.0)
    )
    .withColumn("price", F.round(F.lit(520.0) * F.col("_seasonal_price") * F.col("_iran_spike") + F.randn(seed=609) * 60.0, 2))
    .withColumn("price", F.greatest(F.col("price"), F.lit(200.0)))
    .withColumn("notional_value", F.round(F.col("price") * F.col("volume"), 2))
    .withColumn("delivery_start", F.date_add("_trade_date", (F.rand(seed=610) * 45).cast("int") + 7))
    .withColumn("delivery_end", F.date_add("delivery_start", 30))
    .withColumn("status",
        F.when(F.rand(seed=611) < 0.90, "CONFIRMED")
         .when(F.rand(seed=611) < 0.97, "SETTLED")
         .otherwise("PENDING")
    )
    .drop("id", "_trade_date", "_trade_hour", "_seasonal_vol", "_seasonal_price", "_iran_spike")
)

# --- Manta Resources: B.Rock Van Guard dumps Power, Gas and LNG far below market ---
# APAC multi-commodity trader with grossly oversized clip sizes (~10x typical) and
# sells 35-50% below the prevailing curve — and dumps even harder during the
# Iran-Israel conflict spikes when other desks are scrambling to buy. A textbook
# predatory "panic-the-market" pattern: he sells volume when prices are high,
# depressing the curve and making other longs bleed.
MANTA_TRADES = 800
SLNG_DP_KEY = len(delivery_data)  # Singapore LNG terminal — added last in delivery_data

df_manta = (
    spark.range(0, MANTA_TRADES)
    .withColumn("trade_id", F.concat(F.lit("MANTA-"), F.lpad(F.col("id").cast("string"), 6, "0")))
    # Date sampling: 50% uniform, 50% biased into Iran-conflict spike windows
    # so his footprint is heavily over-represented when others are panic-buying
    .withColumn("_date_class", F.rand(seed=720))
    .withColumn("_uniform_date",
        F.date_add(F.lit(DATE_START), (F.rand(seed=700) * F.datediff(F.lit(DATE_END), F.lit(DATE_START))).cast("int"))
    )
    # Pick one of the four spike windows uniformly, then a date within it
    .withColumn("_spike_window_pick", (F.rand(seed=721) * 4).cast("int"))
    .withColumn("_spike_date",
        F.when(F.col("_spike_window_pick") == 0,  # Apr 2024 — 36 day window
            F.date_add(F.lit("2024-04-10"), (F.rand(seed=722) * 36).cast("int")))
         .when(F.col("_spike_window_pick") == 1,  # Oct 2024 — 51 day window
            F.date_add(F.lit("2024-10-01"), (F.rand(seed=722) * 51).cast("int")))
         .when(F.col("_spike_window_pick") == 2,  # Jun-Sep 2025 — 108 day window
            F.date_add(F.lit("2025-06-15"), (F.rand(seed=722) * 108).cast("int")))
         .otherwise(                              # Jan-Mar 2026 — 55 day window
            F.date_add(F.lit("2026-01-10"), (F.rand(seed=722) * 55).cast("int")))
    )
    .withColumn("_trade_date",
        F.when(F.col("_date_class") < 0.50, F.col("_spike_date"))
         .otherwise(F.col("_uniform_date"))
    )
    .withColumn("_trade_hour", F.lit(7) + (F.rand(seed=701) * 11).cast("int"))
    .withColumn("time_key",
        F.concat(F.date_format("_trade_date", "yyyyMMdd"), F.lpad(F.col("_trade_hour").cast("string"), 2, "0"))
    )
    .withColumn("trade_timestamp",
        F.to_timestamp(F.concat(
            F.col("_trade_date").cast("string"), F.lit(" "),
            F.lpad(F.col("_trade_hour").cast("string"), 2, "0"), F.lit(":"),
            F.lpad((F.rand(seed=702) * 60).cast("int").cast("string"), 2, "0"), F.lit(":"),
            F.lpad((F.rand(seed=703) * 60).cast("int").cast("string"), 2, "0"),
        ))
    )
    .withColumn("trader_key", F.lit(MANTA_TRADER_KEY).cast("long"))
    .withColumn("counterparty_key", (F.rand(seed=704) * num_counterparties).cast("int") + 1)
    # Instrument mix: 30% Power (1-8), 30% Gas (9-14), 40% LNG (21-24)
    .withColumn("_instr_class", F.rand(seed=705))
    .withColumn("instrument_key",
        F.when(F.col("_instr_class") < 0.30, (F.rand(seed=706) * 8).cast("int") + 1)
         .when(F.col("_instr_class") < 0.60, (F.rand(seed=707) * 6).cast("int") + 9)
         .otherwise((F.rand(seed=708) * 4).cast("int") + 21)
    )
    # Delivery points: SLNG-SG primary, plus European hubs for cross-region action
    .withColumn("delivery_point_key",
        F.when(F.rand(seed=709) < 0.50, F.lit(SLNG_DP_KEY))   # Singapore SLNG: 50%
         .when(F.rand(seed=709) < 0.70, F.lit(19))             # TTF: 20%
         .when(F.rand(seed=709) < 0.85, F.lit(20))             # NBP: 15%
         .when(F.rand(seed=709) < 0.95, F.lit(27))             # Gate Rotterdam: 10%
         .otherwise(F.lit(28))                                  # Dragon UK: 5%
    )
    .withColumn("market_key",
        F.element_at(F.array(F.lit(1), F.lit(4), F.lit(5), F.lit(7)),
                     (F.rand(seed=710) * 4).cast("int") + 1)
    )
    # 98% SELL — B.Rock dumps, almost never lifts
    .withColumn("direction", F.when(F.rand(seed=711) < 0.02, "BUY").otherwise("SELL"))
    # Detect if this trade landed in a conflict spike window — drives both
    # the dump intensity and the volume aggression below
    .withColumn("_in_spike",
        ((F.col("_trade_date") >= F.lit("2024-04-10")) & (F.col("_trade_date") <= F.lit("2024-05-15"))) |
        ((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20"))) |
        ((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30"))) |
        ((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")))
    )
    # Grossly oversized clip sizes — ~10x standard baseline, +30% during spikes
    .withColumn("_spike_vol_mult", F.when(F.col("_in_spike"), F.lit(1.30)).otherwise(F.lit(1.0)))
    .withColumn("volume",
        F.round(
            (F.when(F.col("instrument_key").between(1, 8), F.rand(seed=712) * 4750 + 250)         # Power: 250-5000 MW
              .when(F.col("instrument_key").between(9, 14), F.rand(seed=712) * 950000 + 50000)   # Gas:   50k-1M MWh
              .otherwise(F.rand(seed=712) * 650000 + 50000))                                      # LNG:   50k-700k MT
            * F.col("_spike_vol_mult"),
            2)
    )
    # Price: market base × season × Iran-spike × dump_factor (so the dump is
    # measured against the *actual* prevailing market, not a static base).
    # Outside spikes: dump_factor 0.50-0.65 (35-50% below market)
    # During spikes: dump_factor 0.40-0.52 (48-60% below market) — predatory
    .withColumn("_market_base", F.expr(price_base_expr))
    .withColumn("_seasonal_factor",
        F.when(F.month("_trade_date").isin(12, 1, 2), 1.35)
         .when(F.month("_trade_date").isin(11, 3), 1.15)
         .when(F.month("_trade_date").isin(6, 7, 8), 0.80)
         .otherwise(1.0)
    )
    .withColumn("_iran_spike",
        F.when(F.col("instrument_key").between(9, 16) | F.col("instrument_key").between(21, 24),
            F.when((F.col("_trade_date") >= F.lit("2024-04-10")) & (F.col("_trade_date") <= F.lit("2024-05-15")), 1.18)
             .when((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20")), 1.22)
             .when((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30")), 1.28)
             .when((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")), 1.20)
             .otherwise(1.0))
         .when(F.col("instrument_key").between(1, 8),
            F.when((F.col("_trade_date") >= F.lit("2024-10-01")) & (F.col("_trade_date") <= F.lit("2024-11-20")), 1.10)
             .when((F.col("_trade_date") >= F.lit("2025-06-15")) & (F.col("_trade_date") <= F.lit("2025-09-30")), 1.15)
             .when((F.col("_trade_date") >= F.lit("2026-01-10")) & (F.col("_trade_date") <= F.lit("2026-03-05")), 1.08)
             .otherwise(1.0))
         .otherwise(1.0)
    )
    .withColumn("_dump_factor",
        F.when(F.col("_in_spike"), F.rand(seed=713) * 0.12 + 0.40)   # 0.40-0.52 (48-60% below)
         .otherwise(F.rand(seed=713) * 0.15 + 0.50)                    # 0.50-0.65 (35-50% below)
    )
    .withColumn("price",
        F.round(
            F.col("_market_base") * F.col("_seasonal_factor") * F.col("_iran_spike") * F.col("_dump_factor")
            + F.randn(seed=714) * 4.0,
            2)
    )
    .withColumn("price", F.greatest(F.col("price"), F.lit(0.01)))
    .withColumn("notional_value", F.round(F.col("price") * F.col("volume"), 2))
    .withColumn("delivery_start", F.date_add("_trade_date", (F.rand(seed=715) * 30).cast("int") + 1))
    .withColumn("delivery_end", F.date_add("delivery_start",
        F.when(F.col("market_key").isin(1, 2), 1)
         .when(F.col("market_key") == 3, 7)
         .when(F.col("market_key") == 4, 30)
         .when(F.col("market_key") == 5, 90)
         .when(F.col("market_key") == 6, 365)
         .otherwise(30)
    ))
    .withColumn("status",
        F.when(F.rand(seed=716) < 0.85, "CONFIRMED")
         .when(F.rand(seed=716) < 0.95, "SETTLED")
         .otherwise("PENDING")
    )
    .drop("id", "_date_class", "_uniform_date", "_spike_window_pick", "_spike_date",
          "_trade_date", "_trade_hour", "_instr_class", "_in_spike", "_spike_vol_mult",
          "_market_base", "_seasonal_factor", "_iran_spike", "_dump_factor")
)

# Union all trade blocks
df_trades = df_trades.unionByName(df_equinor_lng).unionByName(df_kong_lng).unionByName(df_manta)

df_trades.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_trade")
print(f"fact_trade: {df_trades.count():,} rows (incl. {EQUINOR_LNG_TRADES:,} Equinor LNG + {KONG_LNG_TRADES:,} Kong LNG + {MANTA_TRADES:,} Manta dump trades)")

fact_trade: 1,001,900 rows (incl. 900 Equinor LNG + 200 Kong LNG + 800 Manta dump trades)


In [0]:
df_time_slim = spark.table("dim_time").select("time_key", "cal_date", "hour_of_day", "month", "is_peak")
df_dps = spark.table("dim_delivery_point").select("delivery_point_key", "zone_code", "commodity_class")

df_price_grid = df_time_slim.crossJoin(df_dps)

df_market_prices = (
    df_price_grid
    # Base prices calibrated to Yahoo Finance (Brent BZ=F, TTF=F, JKM=F)
    # and published market averages where Yahoo doesn't carry the series.
    .withColumn("_base",
        F.when(F.col("commodity_class") == "Power", 95.0)    # no Yahoo ticker — market estimate
         .when(F.col("commodity_class") == "Gas", 32.0)      # Yahoo TTF=F mean €36 / 1.16 mult
         .when(F.col("commodity_class") == "Oil", 65.0)      # Yahoo BZ=F mean $76 / 1.16 mult
         .when(F.col("commodity_class") == "Carbon", 62.0)   # published EUA mean €72 / 1.16 mult
         .when(F.col("commodity_class") == "LNG", 560.0)     # Yahoo JKM=F mean $650/MT / 1.16 mult
         .otherwise(40.0)
    )
    # Commodity-specific seasonal price factors
    .withColumn("_season",
        F.when(F.col("commodity_class").isin("Power", "Gas", "LNG"),
            F.when(F.col("month").isin(12, 1, 2), 1.45)       # Deep winter: power/gas spike
             .when(F.col("month").isin(11, 3), 1.20)
             .when(F.col("month").isin(6, 7, 8), 0.70)         # Summer: demand drops
             .otherwise(1.0)
        ).otherwise(
            F.when(F.col("month").isin(12, 1, 2), 1.10)        # Oil/carbon: mild seasonality
             .when(F.col("month").isin(6, 7, 8), 0.95)
             .otherwise(1.0)
        )
    )
    .withColumn("_peak_adj",
        F.when((F.col("commodity_class") == "Power") & F.col("is_peak"), 1.35)
         .otherwise(1.0)
    )
    # Iran-Israel conflict event spikes — applied to Gas/Oil/LNG fully,
    # Power partially (follow-through from gas), Carbon/GOs unaffected
    .withColumn("_iran_spike",
        F.when(F.col("commodity_class").isin("Gas", "Oil", "LNG"),
            F.when((F.col("cal_date") >= F.lit("2024-04-10")) & (F.col("cal_date") <= F.lit("2024-05-15")), 1.18)
             .when((F.col("cal_date") >= F.lit("2024-10-01")) & (F.col("cal_date") <= F.lit("2024-11-20")), 1.22)
             .when((F.col("cal_date") >= F.lit("2025-06-15")) & (F.col("cal_date") <= F.lit("2025-09-30")), 1.28)
             .when((F.col("cal_date") >= F.lit("2026-01-10")) & (F.col("cal_date") <= F.lit("2026-03-05")), 1.20)
             .otherwise(1.0))
         .when(F.col("commodity_class") == "Power",
            F.when((F.col("cal_date") >= F.lit("2024-10-01")) & (F.col("cal_date") <= F.lit("2024-11-20")), 1.10)
             .when((F.col("cal_date") >= F.lit("2025-06-15")) & (F.col("cal_date") <= F.lit("2025-09-30")), 1.15)
             .when((F.col("cal_date") >= F.lit("2026-01-10")) & (F.col("cal_date") <= F.lit("2026-03-05")), 1.08)
             .otherwise(1.0))
         .otherwise(1.0)
    )
    .withColumn("price",
        F.round(F.col("_base") * F.col("_season") * F.col("_peak_adj") * F.col("_iran_spike") + F.randn() * 8.0, 2)
    )
    .withColumn("price", F.greatest(F.col("price"), F.lit(0.01)))
    # Market volume also seasonal — winter sees much higher traded volumes
    .withColumn("_vol_season",
        F.when(F.col("commodity_class").isin("Power", "Gas", "LNG"),
            F.when(F.col("month").isin(12, 1, 2), 1.6)
             .when(F.col("month").isin(11, 3), 1.3)
             .when(F.col("month").isin(6, 7, 8), 0.5)
             .otherwise(1.0)
        ).otherwise(1.0)
    )
    .withColumn("volume_traded", F.round((F.rand() * 50000 + 1000) * F.col("_vol_season"), 0).cast("long"))
    .select("time_key", "delivery_point_key", "price", "volume_traded")
)

df_market_prices.write.mode("overwrite").saveAsTable("fact_market_price")
print(f"fact_market_price: {df_market_prices.count():,} rows")

fact_market_price: 744,192 rows


In [0]:
df_positions = (
    spark.table("fact_trade")
    # Derive day-level time_key (hour 00) for daily aggregation
    .withColumn("position_time_key", F.concat(F.substring("time_key", 1, 8), F.lit("00")))
    .withColumn("signed_volume",
        F.when(F.col("direction") == "BUY", F.col("volume"))
         .otherwise(-F.col("volume"))
    )
    .groupBy("position_time_key", "trader_key", "instrument_key", "delivery_point_key")
    .agg(
        F.sum("signed_volume").alias("net_volume"),
        F.sum(F.abs(F.col("volume"))).alias("gross_volume"),
        F.count("*").alias("trade_count"),
        F.avg("price").alias("avg_price"),
        F.sum("notional_value").alias("total_notional"),
        F.sum(F.when(F.col("direction") == "BUY", F.col("volume")).otherwise(0)).alias("buy_volume"),
        F.sum(F.when(F.col("direction") == "SELL", F.col("volume")).otherwise(0)).alias("sell_volume"),
    )
    .withColumnRenamed("position_time_key", "time_key")
    .withColumn("net_volume", F.round("net_volume", 2))
    .withColumn("gross_volume", F.round("gross_volume", 2))
    .withColumn("avg_price", F.round("avg_price", 2))
    .withColumn("total_notional", F.round("total_notional", 2))
    .withColumn("buy_volume", F.round("buy_volume", 2))
    .withColumn("sell_volume", F.round("sell_volume", 2))
    .withColumn("position_type",
        F.when(F.col("net_volume") > 0, "LONG")
         .when(F.col("net_volume") < 0, "SHORT")
         .otherwise("FLAT")
    )
)

df_positions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_position")
print(f"fact_position: {df_positions.count():,} rows")

fact_position: 996,343 rows


## 3. Verification

In [0]:
tables = ["dim_time", "dim_geography", "dim_trader", "dim_counterparty",
          "dim_instrument", "dim_delivery_point", "dim_market",
          "fact_trade", "fact_position", "fact_market_price"]

for t in tables:
    count = spark.table(t).count()
    print(f"  {t:30s} {count:>12,} rows")

  dim_time                             23,256 rows
  dim_geography                            16 rows
  dim_trader                              122 rows
  dim_counterparty                         32 rows
  dim_instrument                           24 rows
  dim_delivery_point                       32 rows
  dim_market                               10 rows
  fact_trade                        1,001,900 rows
  fact_position                       996,343 rows
  fact_market_price                   744,192 rows


In [0]:
%sql
-- Top 10 traders by notional value
SELECT
    t.company,
    t.trader_name,
    t.desk,
    COUNT(*) AS num_trades,
    ROUND(SUM(ft.notional_value), 0) AS total_notional,
    ROUND(AVG(ft.price), 2) AS avg_price
FROM fact_trade ft
JOIN dim_trader t ON ft.trader_key = t.trader_key
GROUP BY t.company, t.trader_name, t.desk
ORDER BY total_notional DESC
LIMIT 10

company,trader_name,desk,num_trades,total_notional,avg_price
BP Trading,Henrik Petersen,LNG-Global,8420,2.99173450989E11,160.71
BP Trading,Anders Virtanen,Power-UK,8257,2.92419633728E11,157.69
BP Trading,Johan Virtanen,Oil-Brent,8226,2.88943485929E11,158.65
BP Trading,Niels Hansen,LNG-Global,8214,2.879169458E11,155.72
Kong Corporation,King Kong,LNG-Global,8341,2.86509224819E11,166.48
BP Trading,Axel Sørensen,Carbon-EUA,8306,2.85378054452E11,157.01
BP Trading,Filip Hansen,Gas-TTF,8126,2.79400693439E11,157.02
Shell Energy,Freya Petersen,Oil-Brent,8157,2.79202120257E11,155.44
BP Trading,Olof Johansson,Power-Nordic,8116,2.76401931555E11,153.92
Shell Energy,Oscar Dupont,Carbon-EUA,8136,2.74040222138E11,155.21


In [0]:
%sql
-- Monthly trading volume by commodity, using dim_time
SELECT
    tm.year,
    tm.month_name,
    tm.month,
    i.commodity,
    COUNT(*) AS num_trades,
    ROUND(SUM(ft.volume), 0) AS total_volume,
    ROUND(AVG(ft.price), 2) AS avg_price
FROM fact_trade ft
JOIN dim_time tm ON ft.time_key = tm.time_key
JOIN dim_instrument i ON ft.instrument_key = i.instrument_key
GROUP BY tm.year, tm.month_name, tm.month, i.commodity
ORDER BY tm.year, tm.month, i.commodity

year,month_name,month,commodity,num_trades,total_volume,avg_price
2024,January,1,Carbon,2595,1.47082379E8,85.11
2024,January,1,Crude Oil,2674,3.1994718E8,85.78
2024,January,1,Guarantees of Origin,2717,7.5784939E7,4.47
2024,January,1,LNG,5343,7.94237992E8,700.15
2024,January,1,Natural Gas,8286,1.720063995E9,67.68
2024,January,1,Power,10668,1.1049761E7,133.48
2024,February,2,Carbon,2425,1.37367991E8,84.9
2024,February,2,Crude Oil,2469,2.90301955E8,85.69
2024,February,2,Guarantees of Origin,2492,7.0551048E7,4.44
2024,February,2,LNG,4983,7.15306426E8,698.65


In [0]:
%sql
-- Average hourly power price by season, peak/off-peak, country and continent
SELECT
    g.continent,
    g.country_name,
    tm.trading_season,
    tm.period_type,
    dp.zone_code,
    ROUND(AVG(mp.price), 2) AS avg_price,
    ROUND(STDDEV(mp.price), 2) AS price_stddev,
    SUM(mp.volume_traded) AS total_volume
FROM fact_market_price mp
JOIN dim_time tm ON mp.time_key = tm.time_key
JOIN dim_delivery_point dp ON mp.delivery_point_key = dp.delivery_point_key
JOIN dim_geography g ON dp.geography_key = g.geography_key
WHERE dp.commodity_class = 'Power'
GROUP BY g.continent, g.country_name, tm.trading_season, tm.period_type, dp.zone_code
ORDER BY g.continent, g.country_name, dp.zone_code, tm.trading_season, tm.period_type

continent,country_name,trading_season,period_type,zone_code,avg_price,price_stddev,total_volume
Europe,Belgium,Summer,Off-Peak,BE,82.28,16.51,108274336
Europe,Belgium,Summer,Peak,BE,111.15,21.01,127320312
Europe,Belgium,Winter,Off-Peak,BE,126.48,17.82,182451465
Europe,Belgium,Winter,Peak,BE,170.7,22.95,217250102
Europe,Denmark,Summer,Off-Peak,DK1,82.3,16.51,106615447
Europe,Denmark,Summer,Peak,DK1,111.02,21.08,127330133
Europe,Denmark,Winter,Off-Peak,DK1,126.55,17.81,185860648
Europe,Denmark,Winter,Peak,DK1,170.5,22.91,219324466
Europe,Denmark,Summer,Off-Peak,DK2,82.37,16.46,106877590
Europe,Denmark,Summer,Peak,DK2,111.09,20.92,128427319


In [0]:
%sql
-- Daily P&L proxy by country
SELECT
    tm.cal_date,
    g.continent,
    g.country_name,
    tr.desk,
    i.commodity,
    SUM(fp.net_volume) AS net_position,
    ROUND(AVG(fp.avg_price), 2) AS avg_entry_price,
    ROUND(SUM(fp.total_notional), 0) AS notional_exposure
FROM fact_position fp
JOIN dim_time tm ON fp.time_key = tm.time_key
JOIN dim_trader tr ON fp.trader_key = tr.trader_key
JOIN dim_instrument i ON fp.instrument_key = i.instrument_key
JOIN dim_delivery_point dp ON fp.delivery_point_key = dp.delivery_point_key
JOIN dim_geography g ON dp.geography_key = g.geography_key
GROUP BY tm.cal_date, g.continent, g.country_name, tr.desk, i.commodity
ORDER BY tm.cal_date DESC, notional_exposure DESC
LIMIT 20

cal_date,continent,country_name,desk,commodity,net_position,avg_entry_price,notional_exposure
2026-08-25,Europe,United Kingdom,LNG-Global,LNG,-323677.39,430.88,2.97543383E8
2026-08-25,Europe,Norway,LNG-Global,LNG,237638.45,417.64,2.33697231E8
2026-08-25,Europe,Netherlands,Carbon-EUA,LNG,273732.34,444.8,1.51620248E8
2026-08-25,Europe,Sweden,Carbon-EUA,LNG,134640.38,483.8,1.47059251E8
2026-08-25,Europe,Sweden,LNG-Global,LNG,-182059.78999999998,456.77,1.44870122E8
2026-08-25,Europe,Norway,Power-CWE,LNG,143528.86000000002,476.57,1.42027541E8
2026-08-25,Europe,Germany,Carbon-EUA,LNG,225570.7,579.98,1.33520797E8
2026-08-25,Europe,Denmark,Power-UK,LNG,318398.07,431.34,1.31112967E8
2026-08-25,Europe,Norway,Gas-TTF,LNG,226863.65999999997,407.27,1.22693225E8
2026-08-25,Europe,Netherlands,Oil-Brent,LNG,-52452.99000000002,471.39,1.18231995E8


In [0]:
%sql
-- Volume breakdown by continent and country
SELECT
    g.continent,
    g.country_code,
    g.country_name,
    COUNT(*) AS num_trades,
    ROUND(SUM(ft.volume), 0) AS total_volume,
    ROUND(SUM(ft.notional_value), 0) AS total_notional
FROM fact_trade ft
JOIN dim_delivery_point dp ON ft.delivery_point_key = dp.delivery_point_key
JOIN dim_geography g ON dp.geography_key = g.geography_key
GROUP BY g.continent, g.country_code, g.country_name
ORDER BY g.continent, total_notional DESC

continent,country_code,country_name,num_trades,total_volume,total_notional
Asia,SG,Singapore,31895,2.222775566E9,4.29210987232E11
Europe,NO,Norway,219528,1.4592485842E10,2.822624042264E12
Europe,UK,United Kingdom,155980,1.0416017092E10,2.021931604162E12
Europe,SE,Sweden,125278,8.369452292E9,1.607025364755E12
Europe,NL,Netherlands,94479,6.413266906E9,1.231631739447E12
Europe,DE,Germany,93204,6.177655167E9,1.205743118906E12
Europe,BE,Belgium,62451,4.272627176E9,8.25139955878E11
Europe,FR,France,62777,4.177350152E9,8.05766787538E11
Europe,DK,Denmark,62541,4.127617291E9,7.91967010309E11
Europe,FI,Finland,31431,2.100273633E9,4.09755691856E11


In [0]:
%sql
-- Confirms B.Rock's behaviour:
--   • price_pct_vs_market should sit between roughly -35% and -55% per instrument
--     (deepest during the Iran-Israel conflict spike windows, where he dumps even harder
--      while other desks are panic-buying)
--   • clip_size_x_vs_market should be ~10x or higher
WITH manta AS (
    SELECT
        i.commodity,
        i.instrument_code,
        ft.instrument_key,
        COUNT(*) AS trade_count,
        ROUND(SUM(ft.volume), 0) AS total_volume,
        ROUND(AVG(ft.volume), 0) AS avg_clip_size,
        ROUND(AVG(ft.price), 2) AS avg_sell_price
    FROM fact_trade ft
    JOIN dim_trader t ON ft.trader_key = t.trader_key
    JOIN dim_instrument i ON ft.instrument_key = i.instrument_key
    WHERE t.trader_name = 'B.Rock Van Guard'
      AND ft.direction = 'SELL'
    GROUP BY i.commodity, i.instrument_code, ft.instrument_key
),
market AS (
    SELECT
        ft.instrument_key,
        ROUND(AVG(ft.price), 2) AS market_avg_price,
        ROUND(AVG(ft.volume), 0) AS market_avg_clip
    FROM fact_trade ft
    JOIN dim_trader t ON ft.trader_key = t.trader_key
    WHERE t.trader_name <> 'B.Rock Van Guard'
    GROUP BY ft.instrument_key
)
SELECT
    m.commodity,
    m.instrument_code,
    m.trade_count,
    m.total_volume,
    m.avg_clip_size,
    mk.market_avg_clip,
    ROUND(m.avg_clip_size / mk.market_avg_clip, 1) AS clip_size_x_vs_market,
    m.avg_sell_price,
    mk.market_avg_price,
    ROUND((m.avg_sell_price - mk.market_avg_price) / mk.market_avg_price * 100, 1) AS price_pct_vs_market
FROM manta m
JOIN market mk ON m.instrument_key = mk.instrument_key
ORDER BY price_pct_vs_market ASC

commodity,instrument_code,trade_count,total_volume,avg_clip_size,market_avg_clip,clip_size_x_vs_market,avg_sell_price,market_avg_price,price_pct_vs_market
LNG,LNG-TTF-SP,236,4.8138311E7,203976.0,95080.0,2.1,443.73,529.01,-16.1
LNG,LNG-JKM-FU,257,5.4262466E7,211138.0,94991.0,2.2,527.2,613.44,-14.1
LNG,LNG-JKM-SP,241,4.8236186E7,200150.0,95630.0,2.1,537.85,625.43,-14.0
Natural Gas,GAS-TTF-QT,210,5.0833296E7,242063.0,134133.0,1.8,27.33,31.2,-12.4
LNG,LNG-DES-NWE,271,4.8694609E7,179685.0,95568.0,1.9,476.92,542.0,-12.0
Power,PWR-BASE-YR,187,217431.0,1163.0,670.0,1.7,76.25,86.42,-11.8
Natural Gas,GAS-TTF-MO,197,4.7226171E7,239727.0,134884.0,1.8,30.11,33.39,-9.8
Power,PWR-BASE-DA,186,185913.0,1000.0,673.0,1.5,92.63,102.58,-9.7
Power,PWR-PEAK-MO,230,224957.0,978.0,662.0,1.5,110.93,120.93,-8.3
Natural Gas,GAS-TTF-YR,206,4.6116782E7,223868.0,133916.0,1.7,26.64,28.98,-8.1


In [0]:
%sql
-- The killer chart for the workshop: shows B.Rock's price discount and
-- clip size split by whether the trade landed inside an Iran-Israel conflict
-- spike window. Expectation:
--   • IN_SPIKE rows have a deeper price discount than NORMAL rows
--   • IN_SPIKE rows have ~30% larger average clip size
-- That is the counter-cyclical "panic-the-market" signature.
WITH classified AS (
    SELECT
        ft.*,
        TO_DATE(SUBSTRING(ft.time_key, 1, 8), 'yyyyMMdd') AS trade_date,
        CASE
            WHEN TO_DATE(SUBSTRING(ft.time_key, 1, 8), 'yyyyMMdd') BETWEEN DATE '2024-04-10' AND DATE '2024-05-15' THEN 'IN_SPIKE'
            WHEN TO_DATE(SUBSTRING(ft.time_key, 1, 8), 'yyyyMMdd') BETWEEN DATE '2024-10-01' AND DATE '2024-11-20' THEN 'IN_SPIKE'
            WHEN TO_DATE(SUBSTRING(ft.time_key, 1, 8), 'yyyyMMdd') BETWEEN DATE '2025-06-15' AND DATE '2025-09-30' THEN 'IN_SPIKE'
            WHEN TO_DATE(SUBSTRING(ft.time_key, 1, 8), 'yyyyMMdd') BETWEEN DATE '2026-01-10' AND DATE '2026-03-05' THEN 'IN_SPIKE'
            ELSE 'NORMAL'
        END AS regime
    FROM fact_trade ft
),
manta AS (
    SELECT
        c.regime,
        i.commodity,
        COUNT(*) AS trade_count,
        ROUND(AVG(c.volume), 0) AS manta_avg_clip,
        ROUND(AVG(c.price), 2) AS manta_avg_price
    FROM classified c
    JOIN dim_trader t ON c.trader_key = t.trader_key
    JOIN dim_instrument i ON c.instrument_key = i.instrument_key
    WHERE t.trader_name = 'B.Rock Van Guard' AND c.direction = 'SELL'
    GROUP BY c.regime, i.commodity
),
market AS (
    SELECT
        c.regime,
        i.commodity,
        ROUND(AVG(c.price), 2) AS market_avg_price,
        ROUND(AVG(c.volume), 0) AS market_avg_clip
    FROM classified c
    JOIN dim_trader t ON c.trader_key = t.trader_key
    JOIN dim_instrument i ON c.instrument_key = i.instrument_key
    WHERE t.trader_name <> 'B.Rock Van Guard'
    GROUP BY c.regime, i.commodity
)
SELECT
    m.commodity,
    m.regime,
    m.trade_count,
    m.manta_avg_clip,
    mk.market_avg_clip,
    ROUND(m.manta_avg_clip / mk.market_avg_clip, 1) AS clip_size_x_vs_market,
    m.manta_avg_price,
    mk.market_avg_price,
    ROUND((m.manta_avg_price - mk.market_avg_price) / mk.market_avg_price * 100, 1) AS price_pct_vs_market
FROM manta m
JOIN market mk ON m.commodity = mk.commodity AND m.regime = mk.regime
ORDER BY m.commodity, m.regime

commodity,regime,trade_count,manta_avg_clip,market_avg_clip,clip_size_x_vs_market,manta_avg_price,market_avg_price,price_pct_vs_market
Carbon,IN_SPIKE,88,57082.0,55526.0,1.0,63.75,64.4,-1.0
Carbon,NORMAL,238,56646.0,55496.0,1.0,67.75,67.07,1.0
Crude Oil,IN_SPIKE,81,113837.0,120340.0,0.9,84.35,79.97,5.5
Crude Oil,NORMAL,241,123170.0,119779.0,1.0,67.49,67.65,-0.2
Guarantees of Origin,IN_SPIKE,92,29301.0,27731.0,1.1,3.27,3.39,-3.5
Guarantees of Origin,NORMAL,248,29006.0,27689.0,1.0,3.42,3.53,-3.1
LNG,IN_SPIKE,405,270532.0,90203.0,3.0,479.23,651.35,-26.4
LNG,NORMAL,600,149610.0,97105.0,1.5,508.32,551.48,-7.8
Natural Gas,IN_SPIKE,400,336289.0,126725.0,2.7,51.73,62.83,-17.7
Natural Gas,NORMAL,849,184422.0,137350.0,1.3,52.49,53.59,-2.1


---
# Part 2 — Primary Key & Foreign Key Constraints

Uses the `catalog` / `schema` widgets defined in Part 1.

# Primary Key & Foreign Key Constraints
Configurable via the **`catalog`** and **`schema`** widgets above (defaults: `classic_stable_mags1.energy_trading`).

## Primary Keys on Dimension Tables

In [0]:
%sql
-- Set PK columns to NOT NULL
ALTER TABLE ${catalog}.${schema}.dim_counterparty ALTER COLUMN counterparty_key SET NOT NULL;
ALTER TABLE ${catalog}.${schema}.dim_delivery_point ALTER COLUMN delivery_point_key SET NOT NULL;
ALTER TABLE ${catalog}.${schema}.dim_geography ALTER COLUMN geography_key SET NOT NULL;
ALTER TABLE ${catalog}.${schema}.dim_instrument ALTER COLUMN instrument_key SET NOT NULL;
ALTER TABLE ${catalog}.${schema}.dim_market ALTER COLUMN market_key SET NOT NULL;
ALTER TABLE ${catalog}.${schema}.dim_time ALTER COLUMN time_key SET NOT NULL;
ALTER TABLE ${catalog}.${schema}.dim_trader ALTER COLUMN trader_key SET NOT NULL;

-- Drop existing constraints if any, cascading to dependent FKs (makes this cell idempotent)
ALTER TABLE ${catalog}.${schema}.dim_counterparty DROP CONSTRAINT IF EXISTS pk_dim_counterparty CASCADE;
ALTER TABLE ${catalog}.${schema}.dim_delivery_point DROP CONSTRAINT IF EXISTS pk_dim_delivery_point CASCADE;
ALTER TABLE ${catalog}.${schema}.dim_geography DROP CONSTRAINT IF EXISTS pk_dim_geography CASCADE;
ALTER TABLE ${catalog}.${schema}.dim_instrument DROP CONSTRAINT IF EXISTS pk_dim_instrument CASCADE;
ALTER TABLE ${catalog}.${schema}.dim_market DROP CONSTRAINT IF EXISTS pk_dim_market CASCADE;
ALTER TABLE ${catalog}.${schema}.dim_time DROP CONSTRAINT IF EXISTS pk_dim_time CASCADE;
ALTER TABLE ${catalog}.${schema}.dim_trader DROP CONSTRAINT IF EXISTS pk_dim_trader CASCADE;

-- Add Primary Key constraints
ALTER TABLE ${catalog}.${schema}.dim_counterparty ADD CONSTRAINT pk_dim_counterparty PRIMARY KEY (counterparty_key);
ALTER TABLE ${catalog}.${schema}.dim_delivery_point ADD CONSTRAINT pk_dim_delivery_point PRIMARY KEY (delivery_point_key);
ALTER TABLE ${catalog}.${schema}.dim_geography ADD CONSTRAINT pk_dim_geography PRIMARY KEY (geography_key);
ALTER TABLE ${catalog}.${schema}.dim_instrument ADD CONSTRAINT pk_dim_instrument PRIMARY KEY (instrument_key);
ALTER TABLE ${catalog}.${schema}.dim_market ADD CONSTRAINT pk_dim_market PRIMARY KEY (market_key);
ALTER TABLE ${catalog}.${schema}.dim_time ADD CONSTRAINT pk_dim_time PRIMARY KEY (time_key);
ALTER TABLE ${catalog}.${schema}.dim_trader ADD CONSTRAINT pk_dim_trader PRIMARY KEY (trader_key);

## Dimension-to-Dimension Foreign Key

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.dim_delivery_point ADD CONSTRAINT fk_delivery_point_geography FOREIGN KEY (geography_key) REFERENCES ${catalog}.${schema}.dim_geography(geography_key);

## Foreign Keys on fact_market_price

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.fact_market_price ADD CONSTRAINT fk_market_price_time FOREIGN KEY (time_key) REFERENCES ${catalog}.${schema}.dim_time(time_key);
ALTER TABLE ${catalog}.${schema}.fact_market_price ADD CONSTRAINT fk_market_price_delivery_point FOREIGN KEY (delivery_point_key) REFERENCES ${catalog}.${schema}.dim_delivery_point(delivery_point_key);

## Foreign Keys on fact_position

In [0]:
%sql
-- Enable type widening on fact tables
ALTER TABLE ${catalog}.${schema}.fact_position SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');
ALTER TABLE ${catalog}.${schema}.fact_trade SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

-- fact_position: widen INT columns to BIGINT
ALTER TABLE ${catalog}.${schema}.fact_position ALTER COLUMN trader_key TYPE BIGINT;
ALTER TABLE ${catalog}.${schema}.fact_position ALTER COLUMN instrument_key TYPE BIGINT;
ALTER TABLE ${catalog}.${schema}.fact_position ALTER COLUMN delivery_point_key TYPE BIGINT;

-- fact_trade: widen INT columns to BIGINT
ALTER TABLE ${catalog}.${schema}.fact_trade ALTER COLUMN trader_key TYPE BIGINT;
ALTER TABLE ${catalog}.${schema}.fact_trade ALTER COLUMN counterparty_key TYPE BIGINT;
ALTER TABLE ${catalog}.${schema}.fact_trade ALTER COLUMN instrument_key TYPE BIGINT;
ALTER TABLE ${catalog}.${schema}.fact_trade ALTER COLUMN delivery_point_key TYPE BIGINT;
ALTER TABLE ${catalog}.${schema}.fact_trade ALTER COLUMN market_key TYPE BIGINT;

In [0]:
%sql
-- Drop constraint from earlier partial run
ALTER TABLE ${catalog}.${schema}.fact_position DROP CONSTRAINT IF EXISTS fk_position_time;

ALTER TABLE ${catalog}.${schema}.fact_position ADD CONSTRAINT fk_position_time FOREIGN KEY (time_key) REFERENCES ${catalog}.${schema}.dim_time(time_key);
ALTER TABLE ${catalog}.${schema}.fact_position ADD CONSTRAINT fk_position_trader FOREIGN KEY (trader_key) REFERENCES ${catalog}.${schema}.dim_trader(trader_key);
ALTER TABLE ${catalog}.${schema}.fact_position ADD CONSTRAINT fk_position_instrument FOREIGN KEY (instrument_key) REFERENCES ${catalog}.${schema}.dim_instrument(instrument_key);
ALTER TABLE ${catalog}.${schema}.fact_position ADD CONSTRAINT fk_position_delivery_point FOREIGN KEY (delivery_point_key) REFERENCES ${catalog}.${schema}.dim_delivery_point(delivery_point_key);

## Foreign Keys on fact_trade

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.fact_trade ADD CONSTRAINT fk_trade_time FOREIGN KEY (time_key) REFERENCES ${catalog}.${schema}.dim_time(time_key);
ALTER TABLE ${catalog}.${schema}.fact_trade ADD CONSTRAINT fk_trade_trader FOREIGN KEY (trader_key) REFERENCES ${catalog}.${schema}.dim_trader(trader_key);
ALTER TABLE ${catalog}.${schema}.fact_trade ADD CONSTRAINT fk_trade_counterparty FOREIGN KEY (counterparty_key) REFERENCES ${catalog}.${schema}.dim_counterparty(counterparty_key);
ALTER TABLE ${catalog}.${schema}.fact_trade ADD CONSTRAINT fk_trade_instrument FOREIGN KEY (instrument_key) REFERENCES ${catalog}.${schema}.dim_instrument(instrument_key);
ALTER TABLE ${catalog}.${schema}.fact_trade ADD CONSTRAINT fk_trade_delivery_point FOREIGN KEY (delivery_point_key) REFERENCES ${catalog}.${schema}.dim_delivery_point(delivery_point_key);
ALTER TABLE ${catalog}.${schema}.fact_trade ADD CONSTRAINT fk_trade_market FOREIGN KEY (market_key) REFERENCES ${catalog}.${schema}.dim_market(market_key);

## Verify Constraints

In [0]:
%sql
SELECT table_name, constraint_name, constraint_type
FROM ${catalog}.information_schema.table_constraints
WHERE constraint_schema = '${schema}'
ORDER BY table_name, constraint_type;

table_name,constraint_name,constraint_type
dim_counterparty,pk_dim_counterparty,PRIMARY KEY
dim_delivery_point,fk_delivery_point_geography,FOREIGN KEY
dim_delivery_point,pk_dim_delivery_point,PRIMARY KEY
dim_geography,pk_dim_geography,PRIMARY KEY
dim_instrument,pk_dim_instrument,PRIMARY KEY
dim_market,pk_dim_market,PRIMARY KEY
dim_time,pk_dim_time,PRIMARY KEY
dim_trader,pk_dim_trader,PRIMARY KEY
fact_market_price,fk_market_price_time,FOREIGN KEY
fact_market_price,fk_market_price_delivery_point,FOREIGN KEY


---
# Part 3 — Metric View

Creates the governed metric view over `fact_trade`.

In [0]:
%sql
CREATE OR REPLACE VIEW ${catalog}.${schema}.energy_trading_metrics
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Energy Trading KPIs — comprehensive metric view joining fact_trade with all dimension tables for trading activity analysis, market trends, and trade lifecycle tracking."

source: ${catalog}.${schema}.fact_trade

joins:
  - name: dt
    source: ${catalog}.${schema}.dim_time
    on: source.time_key = dt.time_key
  - name: dtr
    source: ${catalog}.${schema}.dim_trader
    on: source.trader_key = dtr.trader_key
  - name: dcp
    source: ${catalog}.${schema}.dim_counterparty
    on: source.counterparty_key = dcp.counterparty_key
  - name: di
    source: ${catalog}.${schema}.dim_instrument
    on: source.instrument_key = di.instrument_key
  - name: ddp
    source: ${catalog}.${schema}.dim_delivery_point
    on: source.delivery_point_key = ddp.delivery_point_key
    joins:
      - name: dg
        source: ${catalog}.${schema}.dim_geography
        on: ddp.geography_key = dg.geography_key
  - name: dm
    source: ${catalog}.${schema}.dim_market
    on: source.market_key = dm.market_key

dimensions:
  # --- Time dimensions ---
  - name: Trade Date
    expr: dt.cal_date
    comment: "Calendar date of the trade"
  - name: Trade Month
    expr: DATE_TRUNC('MONTH', dt.cal_date)
    comment: "Trade month (first day of the month)"
  - name: Year
    expr: dt.year
    comment: "Calendar year"
  - name: Quarter
    expr: dt.quarter
    comment: "Calendar quarter (1-4)"
  - name: Month
    expr: dt.month_name
    comment: "Month name"
  - name: Fiscal Year
    expr: dt.fiscal_year
    comment: "Fiscal year"
  - name: Fiscal Quarter
    expr: dt.fiscal_quarter
    comment: "Fiscal quarter"
  - name: Trading Season
    expr: dt.trading_season
    comment: "Trading season (e.g. Summer, Winter)"
  - name: Day Name
    expr: dt.day_name
    comment: "Day of the week"
  - name: Is Business Day
    expr: dt.is_business_day
    comment: "Whether the trade occurred on a business day"
  - name: Is Peak
    expr: dt.is_peak
    comment: "Whether the trade occurred during peak hours"
  - name: Hour Label
    expr: dt.hour_label
    comment: "Hour label for intraday analysis"

  # --- Trader dimensions ---
  - name: Trader Name
    expr: dtr.trader_name
    comment: "Name of the trader"
  - name: Trader Company
    expr: dtr.company
    comment: "Trader company name"
  - name: Trader Company Country
    expr: dtr.company_country
    comment: "Country of the trader company"
  - name: Trader Company Type
    expr: dtr.company_type
    comment: "Type of the trader company"
  - name: Trading Desk
    expr: dtr.desk
    comment: "Trading desk"
  - name: Desk Group
    expr: dtr.desk_group
    comment: "Trading desk group"
  - name: Trader Seniority
    expr: dtr.seniority
    comment: "Seniority level of the trader"

  # --- Counterparty dimensions ---
  - name: Counterparty Name
    expr: dcp.counterparty_name
    comment: "Legal name of the counterparty"
  - name: Counterparty Type
    expr: dcp.counterparty_type
    comment: "Classification of the counterparty"
  - name: Counterparty Country
    expr: dcp.country_code
    comment: "Country code of the counterparty"
  - name: Credit Rating
    expr: dcp.credit_rating
    comment: "Counterparty credit rating"
  - name: Is Exchange
    expr: dcp.is_exchange
    comment: "Whether the counterparty is an exchange"

  # --- Instrument dimensions ---
  - name: Instrument Name
    expr: di.instrument_name
    comment: "Name of the traded instrument"
  - name: Instrument Code
    expr: di.instrument_code
    comment: "Code of the traded instrument"
  - name: Commodity
    expr: di.commodity
    comment: "Commodity type (e.g. Power, Gas, Oil)"
  - name: Unit
    expr: di.unit
    comment: "Unit of measure for the instrument"
  - name: Currency
    expr: di.currency
    comment: "Trading currency"
  - name: Is Physical Instrument
    expr: di.is_physical
    comment: "Whether the instrument involves physical delivery"
  - name: Is Financial Instrument
    expr: di.is_financial
    comment: "Whether the instrument is financially settled"

  # --- Delivery Point dimensions ---
  - name: Zone Name
    expr: ddp.zone_name
    comment: "Delivery zone name"
  - name: Zone Code
    expr: ddp.zone_code
    comment: "Delivery zone code"
  - name: Commodity Class
    expr: ddp.commodity_class
    comment: "Commodity class at the delivery point"

  # --- Market dimensions ---
  - name: Market Name
    expr: dm.market_name
    comment: "Name of the market"
  - name: Market Code
    expr: dm.market_code
    comment: "Market code"
  - name: Is OTC
    expr: dm.is_otc
    comment: "Whether the market is over-the-counter"
  - name: Is Physical Market
    expr: dm.is_physical
    comment: "Whether the market involves physical delivery"

  # --- Geography dimensions (nested join through delivery point) ---
  - name: Country Name
    expr: ddp.dg.country_name
    comment: "Country of the delivery point"
  - name: Continent
    expr: ddp.dg.continent
    comment: "Continent of the delivery point"
  - name: Timezone
    expr: ddp.dg.timezone
    comment: "Timezone of the delivery geography"

  # --- Fact degenerate dimensions ---
  - name: Trade ID
    expr: source.trade_id
    comment: "Unique trade identifier"
  - name: Direction
    expr: source.direction
    comment: "Trade direction (BUY/SELL)"
  - name: Status
    expr: source.status
    comment: "Current trade status"
  - name: Delivery Start
    expr: source.delivery_start
    comment: "Start date of the delivery period"
  - name: Delivery End
    expr: source.delivery_end
    comment: "End date of the delivery period"

measures:
  - name: Trade Count
    expr: COUNT(1)
    comment: "Total number of trades"
  - name: Total Volume
    expr: SUM(source.volume)
    comment: "Total traded volume"
  - name: Total Notional Value
    expr: SUM(source.notional_value)
    comment: "Total notional value of all trades"
  - name: Average Price
    expr: AVG(source.price)
    comment: "Average trade price"
  - name: Average Volume
    expr: AVG(source.volume)
    comment: "Average trade volume"
  - name: Average Notional Value
    expr: AVG(source.notional_value)
    comment: "Average notional value per trade"
  - name: Max Price
    expr: MAX(source.price)
    comment: "Maximum trade price"
  - name: Min Price
    expr: MIN(source.price)
    comment: "Minimum trade price"
  - name: Max Volume
    expr: MAX(source.volume)
    comment: "Largest single trade volume"
  - name: Unique Traders
    expr: COUNT(DISTINCT source.trader_key)
    comment: "Number of distinct traders"
  - name: Unique Counterparties
    expr: COUNT(DISTINCT source.counterparty_key)
    comment: "Number of distinct counterparties"
  - name: Unique Instruments
    expr: COUNT(DISTINCT source.instrument_key)
    comment: "Number of distinct instruments traded"
  - name: Buy Volume
    expr: SUM(CASE WHEN source.direction = 'BUY' THEN source.volume ELSE 0 END)
    comment: "Total volume of buy trades"
  - name: Sell Volume
    expr: SUM(CASE WHEN source.direction = 'SELL' THEN source.volume ELSE 0 END)
    comment: "Total volume of sell trades"
  - name: Buy Notional
    expr: SUM(CASE WHEN source.direction = 'BUY' THEN source.notional_value ELSE 0 END)
    comment: "Total notional value of buy trades"
  - name: Sell Notional
    expr: SUM(CASE WHEN source.direction = 'SELL' THEN source.notional_value ELSE 0 END)
    comment: "Total notional value of sell trades"
  - name: VWAP
    expr: SUM(source.price * source.volume) / NULLIF(SUM(source.volume), 0)
    comment: "Volume-weighted average price"
  - name: Trading Profit
    expr: |-
      SUM(CASE
        WHEN source.direction = 'SELL' THEN source.price * source.volume
        WHEN source.direction = 'BUY' THEN -(source.price * source.volume)
        ELSE 0
      END)
    comment: "Net trading profit — sell revenue (price × volume) minus buy cost (price × volume)"
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: Revenue
    expr: SUM(source.price * source.volume) FILTER (WHERE Direction = 'SELL')
    comment: "Total revenue from sell trades (price × volume)"
    display_name: Revenue
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  # --- Rolling window measures ---
  - name: Rolling 12M Trade Count
    expr: MEASURE(`Trade Count`)
    comment: "Total count of trades over a trailing 12-month rolling window"
    window:
      - order: Trade Date
        range: trailing 12 month
        semiadditive: last
  - name: Rolling 6M Trade Count
    expr: MEASURE(`Trade Count`)
    comment: "Total count of trades over a trailing 6-month rolling window"
    window:
      - order: Trade Date
        range: trailing 6 month
        semiadditive: last
  - name: Avg 6M Monthly Trade Count
    expr: MEASURE(`Rolling 6M Trade Count`) / 6
    comment: "Average monthly trade count over a rolling 6-month window"
  - name: Rolling 6M Total Volume
    expr: MEASURE(`Total Volume`)
    comment: "Trailing 6-month rolling total trade volume"
    window:
      - order: Trade Date
        semiadditive: last
        range: trailing 6 month
  - name: Avg 6M Monthly Total Volume
    expr: MEASURE(`Rolling 6M Total Volume`) / 6
    comment: "Average monthly trade volume over trailing 6 months"
  # --- Month-over-month revenue growth ---
  - name: Monthly Revenue
    expr: MEASURE(`Revenue`)
    comment: "Current month revenue"
    window:
      - order: Trade Month
        range: current
        semiadditive: last
  - name: Prev Month Revenue
    expr: MEASURE(`Revenue`)
    comment: "Previous month revenue"
    window:
      - order: Trade Month
        range: trailing 1 month
        semiadditive: last
  - name: MoM Revenue Growth
    expr: (MEASURE(`Monthly Revenue`) - MEASURE(`Prev Month Revenue`)) / NULLIF(MEASURE(`Prev Month Revenue`), 0) * 100
    comment: "Month-over-month revenue growth percentage"
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 2
$$;

-- Query: MoM Revenue Growth by month
SELECT
  `Trade Month`,
  MEASURE(`Monthly Revenue`) AS `Current Month Revenue`,
  MEASURE(`Prev Month Revenue`) AS `Previous Month Revenue`,
  MEASURE(`MoM Revenue Growth`) AS `MoM Growth %`
FROM ${catalog}.${schema}.energy_trading_metrics
GROUP BY `Trade Month`
ORDER BY `Trade Month` ASC;

Trade Month,Current Month Revenue,Previous Month Revenue,MoM Growth %
2024-01-01T00:00:00.000Z,3.293352129030465E11,null,null
2024-02-01T00:00:00.000Z,3.054111634836879E11,3.293352129030465E11,-7.264346016470951
2024-03-01T00:00:00.000Z,2.179262045460933E11,3.054111634836879E11,-28.644977459138367
2024-04-01T00:00:00.000Z,1.8554883114994418E11,2.179262045460933E11,-14.857035418749293
2024-05-01T00:00:00.000Z,1.6036465302634982E11,1.8554883114994418E11,-13.57280343266767
2024-06-01T00:00:00.000Z,6.991740096773677E10,1.6036465302634982E11,-56.40099008835288
2024-07-01T00:00:00.000Z,7.51365016976357E10,6.991740096773677E10,7.46466638871097
2024-08-01T00:00:00.000Z,7.39512121056618E10,7.51365016976357E10,-1.577515009607094
2024-09-01T00:00:00.000Z,1.4068749369881686E11,7.39512121056618E10,90.24366158840233
2024-10-01T00:00:00.000Z,2.0431876071110495E11,1.4068749369881686E11,45.228801323669614


In [0]:
%sql
SELECT
  `Commodity`,
  MEASURE(`Buy Volume`) AS buy_vol,
  MEASURE(`Sell Volume`) AS sell_vol
FROM ${catalog}.${schema}.energy_trading_metrics
GROUP BY `Commodity`
ORDER BY buy_vol + sell_vol DESC

Commodity,buy_vol,sell_vol
Natural Gas,1.774604053206007E10,1.6075242760579967E10
LNG,8.464702500709978E9,7.675940166980008E9
Crude Oil,5.205632497889985E9,4.738073782650008E9
Carbon,2.41323176560999E9,2.2014428589700017E9
Guarantees of Origin,1.2184095625799997E9,1.0906292399300008E9
Power,1.1738234476999989E8,1.0699023806000009E8


---
# Part 4 — AI-Generated Descriptions

Uses `ai_query` to generate 2–5 sentence table descriptions and 2–3 sentence column descriptions for every table and metric view, then applies them as Unity Catalog comments. Leave `dry_run = true` to preview; set `false` to write.

# Energy Trading — AI-Generated Table & Column Descriptions

Uses the built-in **`ai_query`** AI Function to generate documentation for **every
table and metric view** in the dimensional model, then applies it to Unity Catalog:

- **Table / view descriptions** — 2–5 sentences each
- **Column descriptions** — 2–3 sentences each

The model sees each object's column list (with types) plus a few sample rows
(for base tables) or the view definition (for metric views), so the descriptions
are grounded in the actual energy-trading domain rather than generic boilerplate.

### How to run
1. Set the `catalog` / `schema` widgets to the deployed dimensional model.
2. Leave `dry_run = true` for a preview, or set `false` to write the comments.
3. **Run All**. A serverless / Pro SQL warehouse or DBR 15.1+ cluster is required
   for AI Functions.

In [0]:
# DBTITLE 1,Part 4 configuration (AI descriptions)
# catalog / schema are inherited from Part 1's Configuration cell (CATALOG, SCHEMA).
dbutils.widgets.text("model", "databricks-claude-sonnet-4", "AI model endpoint")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry run (preview only)")
dbutils.widgets.dropdown("skip_documented", "false", ["true", "false"], "Skip already-documented objects")

MODEL           = dbutils.widgets.get("model").strip()
DRY_RUN         = dbutils.widgets.get("dry_run") == "true"
SKIP_DOCUMENTED = dbutils.widgets.get("skip_documented") == "true"
DOMAIN          = ("an ENERGY TRADING business (power, gas, oil, and carbon markets: trades, "
                   "positions, market prices, counterparties, delivery points, and instruments)")

print(f"Target:          {CATALOG}.{SCHEMA}")
print(f"Model:           {MODEL}")
print(f"Dry run:         {DRY_RUN}  (set dry_run=false to apply comments)")
print(f"Skip documented: {SKIP_DOCUMENTED}")

Target:          classic_stable_magh.energy_trading
Model:           databricks-claude-sonnet-4
Dry run:         False  (set dry_run=false to apply comments)
Skip documented: True


In [0]:
import json
from pyspark.sql.functions import expr

def sql_lit(text: str) -> str:
    """Collapse whitespace and escape single quotes for a SQL string literal."""
    return " ".join(str(text).split()).replace("'", "''")

def list_objects():
    """Return [(name, table_type, is_view, comment)] for every object in the schema."""
    rows = spark.sql(f"""
        SELECT table_name, table_type, comment
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}'
        ORDER BY table_type, table_name
    """).collect()
    out = []
    for r in rows:
        is_view = "VIEW" in (r["table_type"] or "").upper()
        out.append((r["table_name"], r["table_type"], is_view, r["comment"]))
    return out

def list_columns(table_name: str):
    """Ordered [(column_name, data_type)] for a table/view."""
    rows = spark.sql(f"""
        SELECT column_name, data_type
        FROM {CATALOG}.information_schema.columns
        WHERE table_schema = '{SCHEMA}' AND table_name = '{table_name}'
        ORDER BY ordinal_position
    """).collect()
    return [(r["column_name"], r["data_type"]) for r in rows]

def sample_rows(table_name: str, n: int = 3) -> str:
    """A compact string of up to n sample rows (base tables only)."""
    try:
        pdf = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.`{table_name}` LIMIT {n}").toPandas()
        return pdf.to_json(orient="records")[:2000]
    except Exception as e:
        return f"(no sample available: {e})"

def view_definition(table_name: str) -> str:
    try:
        rows = spark.sql(f"""
            SELECT view_definition
            FROM {CATALOG}.information_schema.views
            WHERE table_schema = '{SCHEMA}' AND table_name = '{table_name}'
        """).collect()
        return (rows[0]["view_definition"] or "")[:3000] if rows else ""
    except Exception:
        return ""

def build_prompt(name, is_view, columns):
    cols_desc = "\n".join(f"  - {c} ({t})" for c, t in columns)
    kind = "metric view" if is_view else "table"
    context = (f"VIEW DEFINITION:\n{view_definition(name)}"
               if is_view else f"SAMPLE ROWS (JSON):\n{sample_rows(name)}")
    col_keys = ", ".join(f'"{c}"' for c, _ in columns)
    return (
        f"You are a data catalog assistant documenting a star-schema data warehouse for {DOMAIN}.\n\n"
        f"Document the {kind} `{CATALOG}.{SCHEMA}.{name}`.\n\n"
        f"COLUMNS:\n{cols_desc}\n\n{context}\n\n"
        "Return ONLY a JSON object with this exact shape and nothing else:\n"
        '{"table_description": "<2 to 5 sentences describing what this object represents, '
        'its grain, and how it is used for analytics and reporting>", '
        f'"columns": {{{col_keys}: "<2 to 3 sentences describing this column, its meaning, '
        'units where relevant, and any keys/relationships>"}}}\n'
        "Rules: table_description must be 2-5 sentences; every column description must be "
        "2-3 sentences; be specific to the domain described above; do not invent columns; "
        "return valid JSON only."
    )

objects = list_objects()
print(f"Found {len(objects)} objects in {CATALOG}.{SCHEMA}:")
for name, ttype, is_view, comment in objects:
    flag = " (documented)" if comment else ""
    print(f"  - {name:24s} [{ttype}]{flag}")

Found 11 objects in classic_stable_magh.energy_trading:
  - dim_counterparty         [MANAGED] (documented)
  - dim_delivery_point       [MANAGED] (documented)
  - dim_geography            [MANAGED] (documented)
  - dim_instrument           [MANAGED] (documented)
  - dim_market               [MANAGED] (documented)
  - dim_time                 [MANAGED] (documented)
  - dim_trader               [MANAGED] (documented)
  - fact_market_price        [MANAGED] (documented)
  - fact_position            [MANAGED] (documented)
  - fact_trade               [MANAGED] (documented)
  - energy_trading_metrics   [METRIC_VIEW] (documented)


In [0]:
targets = [(n, tt, iv) for (n, tt, iv, cmt) in objects if not (SKIP_DOCUMENTED and cmt)]

meta = {}          # name -> {"is_view", "columns"}
prompt_rows = []   # (name, prompt)
raw = {}
if not targets:
    # Skip only THIS section — never dbutils.notebook.exit(), which would halt the
    # entire merged full-solution notebook so later parts (e.g. the Kong GL model)
    # would never run.
    print("Nothing to document (all objects already have comments) — skipping this section.")
else:
    for name, ttype, is_view in targets:
        cols = list_columns(name)
        meta[name] = {"is_view": is_view, "columns": [c for c, _ in cols]}
        prompt_rows.append((name, build_prompt(name, is_view, cols)))

    prompts_df = spark.createDataFrame(prompt_rows, "name STRING, prompt STRING")
    result_df = prompts_df.withColumn(
        "out",
        expr(f"to_json(ai_query('{MODEL}', prompt, failOnError => false))")
    )
    # Materialize once and pull to the driver (object count is small)
    raw = {r["name"]: r["out"] for r in result_df.select("name", "out").collect()}
    print(f"Generated responses for {len(raw)} objects.")

Nothing to document (all objects already have comments) — skipping this section.


In [0]:
import re

def _extract_json(text: str):
    """Try to extract a JSON object from text that may include markdown fences."""
    # Try to find a fenced JSON block first
    m = re.search(r'```(?:json)?\s*\n?(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        return m.group(1)
    # Otherwise find the first { ... } block
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def parse_response(out_json: str):
    """out_json is to_json() of the ai_query struct: {result, errorMessage}."""
    try:
        outer = json.loads(out_json)
    except Exception as e:
        return None, f"could not parse ai_query envelope: {e}"
    err = outer.get("errorMessage") or outer.get("error")
    if err:
        return None, err
    resp = outer.get("result") or outer.get("response")
    if not resp:
        return None, "empty response"
    # Try direct JSON parse first
    try:
        return json.loads(resp), None
    except Exception:
        pass
    # Try extracting JSON from markdown/text response
    extracted = _extract_json(resp)
    if extracted:
        try:
            return json.loads(extracted), None
        except Exception as e:
            return None, f"extracted JSON is invalid: {e}"
    return None, "model did not return valid JSON (no JSON object found in response)"

parsed = {}   # name -> {"table": str, "columns": {col: str}}
errors = {}
for name, out_json in raw.items():
    data, err = parse_response(out_json)
    if err:
        errors[name] = err
        continue
    parsed[name] = {
        "table": (data.get("table_description") or "").strip(),
        "columns": {k: (v or "").strip() for k, v in (data.get("columns") or {}).items()},
    }

if errors:
    print("⚠️  Errors:")
    for n, e in errors.items():
        print(f"   {n}: {e}")
print(f"Parsed descriptions for {len(parsed)} objects.")

Parsed descriptions for 0 objects.


In [0]:
preview = []
for name, d in parsed.items():
    preview.append((name, "TABLE", d["table"]))
    for col in meta[name]["columns"]:
        preview.append((name, col, d["columns"].get(col, "⚠️ (missing — will be skipped)")))

preview_df = spark.createDataFrame(preview, "object STRING, target STRING, proposed_description STRING")
print("Review the proposed descriptions below. Set dry_run=false and re-run to apply.")
display(preview_df)

Review the proposed descriptions below. Set dry_run=false and re-run to apply.


object,target,proposed_description


In [0]:
def apply_comments():
    applied, skipped, failed = 0, 0, []
    for name, d in parsed.items():
        is_view = meta[name]["is_view"]
        fq = f"{CATALOG}.{SCHEMA}.`{name}`"
        obj_kw = "VIEW" if is_view else "TABLE"

        # Object-level comment
        if d["table"]:
            try:
                spark.sql(f"COMMENT ON {obj_kw} {fq} IS '{sql_lit(d['table'])}'")
                applied += 1
            except Exception as e:
                failed.append((name, "TABLE", str(e)[:200]))

        # Column-level comments
        alter_kw = "ALTER VIEW" if is_view else "ALTER TABLE"
        for col in meta[name]["columns"]:
            desc = d["columns"].get(col)
            if not desc:
                skipped += 1
                continue
            try:
                spark.sql(f"{alter_kw} {fq} ALTER COLUMN `{col}` COMMENT '{sql_lit(desc)}'")
                applied += 1
            except Exception as e:
                failed.append((name, col, str(e)[:200]))
    return applied, skipped, failed

if DRY_RUN:
    print("DRY RUN — no comments written. Set the dry_run widget to 'false' and re-run to apply.")
else:
    applied, skipped, failed = apply_comments()
    print(f"✅ Applied {applied} comments.  Skipped {skipped} (no description).")
    if failed:
        print(f"⚠️  {len(failed)} failed (metric-view columns may not accept ALTER COLUMN):")
        for name, tgt, err in failed:
            print(f"   {name}.{tgt}: {err}")

✅ Applied 0 comments.  Skipped 0 (no description).


In [0]:
if not DRY_RUN:
    display(spark.sql(f"""
        SELECT table_name AS object, 'TABLE' AS target, comment AS description
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}' AND comment IS NOT NULL
        UNION ALL
        SELECT table_name, column_name, comment
        FROM {CATALOG}.information_schema.columns
        WHERE table_schema = '{SCHEMA}' AND comment IS NOT NULL
        ORDER BY object, target
    """))

object,target,description
dim_counterparty,TABLE,"This dimension table contains master data for all counterparties involved in energy trading transactions across power, gas, oil, and carbon markets. Each row represents a unique trading counterparty at the grain of one counterparty per record. The table serves as the primary reference for counterparty information in trade booking, risk management, and credit analysis within the energy trading data warehouse."
dim_counterparty,counterparty_id,"This is the business key or natural identifier for each counterparty, typically following a standardized naming convention like 'CP-0001'. It represents the unique identifier used in operational systems to reference trading partners and counterparties."
dim_counterparty,counterparty_key,"This is the surrogate primary key for the counterparty dimension table, used for efficient joins with fact tables. It serves as the foreign key reference in trade and position fact tables to link transactions to specific counterparties."
dim_counterparty,counterparty_name,"This field contains the full legal or trading name of the counterparty organization. It provides the human-readable identification of energy trading partners such as utilities, producers, traders, and financial institutions."
dim_counterparty,counterparty_type,"This attribute categorizes the counterparty by their primary business function in energy markets, such as 'Producer', 'Utility', 'Trader', or 'Financial'. It enables analysis and reporting by counterparty business model and market role."
dim_counterparty,country_code,"This field stores the ISO country code representing the primary jurisdiction or headquarters location of the counterparty. It supports geographic analysis, regulatory reporting, and country-specific risk assessments in energy trading operations."
dim_counterparty,credit_rating,"This attribute contains the credit rating assigned to the counterparty, typically using standard rating scales like 'AAA', 'AA', 'BBB'. It is essential for credit risk management, exposure limits, and counterparty risk analysis in energy trading portfolios."
dim_counterparty,is_exchange,This boolean flag indicates whether the counterparty is an organized exchange or trading platform versus a bilateral trading partner. It distinguishes between exchange-traded transactions and over-the-counter (OTC) bilateral trades for regulatory and operational reporting.
dim_delivery_point,TABLE,"This dimension table contains reference data for delivery points used in energy trading transactions. Each row represents a unique delivery zone or location where commodities like power, gas, or oil can be physically delivered or settled. The grain is one row per delivery point, and it serves as a lookup table to enrich trading facts with geographic and commodity-specific delivery information for position management and settlement analytics."
dim_delivery_point,commodity_class,"Classification of the primary commodity type traded at this delivery point. Common values include Power, Gas, Oil, and Carbon, determining the applicable market rules and settlement mechanisms."


---
# Part 5 — Kong Corporation General Ledger

Builds a GL star schema for Corporate Performance Management in its own `kong_gl` schema (same catalog as Parts 1–4), then adds constraints, a corporate reporting metric view, and AI-generated descriptions:

- **5.1** Dimensions (date, account, cost center, entity, scenario, currency) & facts (journal, monthly balance, plan)
- **5.2** Primary & foreign keys
- **5.3** Corporate reporting metric view (`kong_gl_metrics`)
- **5.4** AI-generated table & column descriptions

Set `gl_schema` / `num_journals` in the Part 5.1 configuration cell.

# Kong Corporation — General Ledger Dimensional Model

A **general-ledger star schema** for **Corporate Performance Management (CPM)** and
financial reporting, built for **Kong Corporation** — the energy producer/trader from
the Energy Trading demo. Same timeframe and fiscal calendar as that demo so the
finance layer lines up with the trading layer.

## Schema Overview

| Table | Type | Description |
|-------|------|-------------|
| `dim_date` | Dimension | Daily calendar with fiscal year (Oct start) & accounting period |
| `dim_account` | Dimension | Chart of accounts: statement → type → category → account |
| `dim_cost_center` | Dimension | Trading desks, generation ops, and corporate functions |
| `dim_entity` | Dimension | Kong group legal entities and functional currencies |
| `dim_scenario` | Dimension | Actual / Budget / Forecast |
| `dim_currency` | Dimension | Currencies and FX rate to the EUR reporting currency |
| `fact_gl_journal` | Fact | Double-entry journal lines (Actuals) — transaction grain |
| `fact_gl_balance` | Fact | Monthly opening / movement / closing balances (Actuals) |
| `fact_financial_plan` | Fact | Budget & Forecast monthly targets for P&L accounts |

### Conventions
- **Signed amounts**: debits are positive, credits are negative, so every journal nets
  to zero and the whole ledger is a guaranteed trial balance (Σ `signed_amount` = 0).
- **Reporting currency** is EUR (`amount_*_rpt`); each entity also posts in its local
  functional currency (`amount_*_lcl`).
- **Fiscal year** starts in October (matches `dim_time` in the trading model).

In [0]:
# DBTITLE 1,Part 5 configuration (Kong GL)
# catalog / date_end are inherited from Part 1. The GL uses its OWN schema, so it
# cannot reuse Part 1's `schema` widget (that one points at energy_trading).
from datetime import date as _date
dbutils.widgets.text("gl_schema", "kong_gl", "GL Schema")
dbutils.widgets.text("num_journals", "120000", "Number of journal entries")

GL_SCHEMA    = dbutils.widgets.get("gl_schema").strip() or "kong_gl"
NUM_JOURNALS = int(dbutils.widgets.get("num_journals").strip() or "120000")
SCHEMA       = GL_SCHEMA          # Part 5 writes to the GL schema, not energy_trading
DATE_START   = "2024-01-01"
DATE_END     = dbutils.widgets.get("date_end").strip() or _date.today().isoformat()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GL_SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {GL_SCHEMA}")

print(f"GL target:    {CATALOG}.{GL_SCHEMA}")
print(f"Date range:   {DATE_START} → {DATE_END}")
print(f"Journals:     {NUM_JOURNALS:,}  (≈ {NUM_JOURNALS*2:,} lines)")

GL target:    classic_stable_magh.kong_gl
Date range:   2024-01-01 → 2026-08-26
Journals:     120,000  (≈ 240,000 lines)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Extend through the END OF THE FINAL MONTH so month-end period keys (used by
# fact_gl_balance / fact_financial_plan) always exist in dim_date.
df_cal = spark.sql(f"""
    SELECT explode(sequence(to_date('{DATE_START}'), last_day(to_date('{DATE_END}')), interval 1 day)) AS cal_date
""")

dim_date = (
    df_cal
    .withColumn("date_key", F.date_format("cal_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("cal_date"))
    .withColumn("quarter", F.quarter("cal_date"))
    .withColumn("month", F.month("cal_date"))
    .withColumn("month_name", F.date_format("cal_date", "MMMM"))
    .withColumn("period", F.date_format("cal_date", "yyyy-MM"))          # accounting period
    .withColumn("day_of_week", F.dayofweek("cal_date"))
    .withColumn("day_name", F.date_format("cal_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("cal_date").isin(1, 7))
    .withColumn("is_business_day", ~F.dayofweek("cal_date").isin(1, 7))
    .withColumn("fiscal_year",
        F.when(F.month("cal_date") >= 10, F.year("cal_date") + 1).otherwise(F.year("cal_date")))
    .withColumn("fiscal_quarter",
        F.when(F.month("cal_date") >= 10,
               F.concat(F.lit("Q"), (((F.month("cal_date") - 10) / 3) + 1).cast("int").cast("string")))
         .otherwise(
               # Oct fiscal start: Jan-Mar=Q2, Apr-Jun=Q3, Jul-Sep=Q4 → floor((m+2)/3)+1
               F.concat(F.lit("Q"), ((((F.month("cal_date") + 2) / 3).cast("int")) + 1).cast("string"))))
    .withColumn("period_end_date", F.last_day("cal_date"))
    .withColumn("is_month_end", F.col("cal_date") == F.last_day("cal_date"))
    .withColumn("is_quarter_end",
        (F.col("cal_date") == F.last_day("cal_date")) & F.month("cal_date").isin(3, 6, 9, 12))
)
dim_date.write.mode("overwrite").saveAsTable("dim_date")
print(f"dim_date: {dim_date.count():,} rows")

dim_date: 974 rows


In [0]:
currencies = [
    # currency_key, code, name, is_reporting, fx_rate_to_eur
    (1, "EUR", "Euro",            True,  1.000),
    (2, "NOK", "Norwegian Krone", False, 0.087),
    (3, "SEK", "Swedish Krona",   False, 0.088),
    (4, "GBP", "British Pound",   False, 1.170),
    (5, "DKK", "Danish Krone",    False, 0.134),
    (6, "USD", "US Dollar",       False, 0.920),
]
dim_currency = spark.createDataFrame(
    currencies, "currency_key INT, currency_code STRING, currency_name STRING, "
                "is_reporting_currency BOOLEAN, fx_rate_to_eur DOUBLE")
dim_currency.write.mode("overwrite").saveAsTable("dim_currency")
print(f"dim_currency: {dim_currency.count()} rows")

dim_currency: 6 rows


In [0]:
entities = [
    # entity_key, code, name, country, region, functional_currency_key, fx_rate, parent_code, is_consolidated
    (1, "E100", "Kong Corporation (Holding)", "XX", "Group",    1, 1.000, None,   True),
    (2, "E200", "Kong Energy Trading AS",     "NO", "Nordics",  2, 0.087, "E100", True),
    (3, "E300", "Kong Power Generation AB",   "SE", "Nordics",  3, 0.088, "E100", True),
    (4, "E400", "Kong LNG Ltd",               "GB", "UK",       4, 1.170, "E100", True),
    (5, "E500", "Kong Renewables ApS",        "DK", "Nordics",  5, 0.134, "E100", True),
    (6, "E600", "Kong Markets Inc",           "US", "Americas", 6, 0.920, "E100", True),
]
dim_entity = spark.createDataFrame(
    entities, "entity_key INT, entity_code STRING, entity_name STRING, country STRING, "
              "region STRING, currency_key INT, fx_rate_to_eur DOUBLE, parent_entity_code STRING, "
              "is_consolidated BOOLEAN")
dim_entity.write.mode("overwrite").saveAsTable("dim_entity")
print(f"dim_entity: {dim_entity.count()} rows")

dim_entity: 6 rows


In [0]:
scenarios = [
    (1, "ACT", "Actual",   "Actual"),
    (2, "BUD", "Budget",   "Plan"),
    (3, "FCT", "Forecast", "Plan"),
]
dim_scenario = spark.createDataFrame(
    scenarios, "scenario_key INT, scenario_code STRING, scenario_name STRING, scenario_type STRING")
dim_scenario.write.mode("overwrite").saveAsTable("dim_scenario")
ACTUAL_KEY, BUDGET_KEY, FORECAST_KEY = 1, 2, 3
print(f"dim_scenario: {dim_scenario.count()} rows")

dim_scenario: 3 rows


In [0]:
cost_centers = [
    # cost_center_key, code, name, function, region, is_front_office
    (1,  "CC-1001", "Power-Nordic Desk",     "Trading",    "Nordics",  True),
    (2,  "CC-1002", "Power-CWE Desk",         "Trading",    "CWE",      True),
    (3,  "CC-1003", "Power-UK Desk",          "Trading",    "UK",       True),
    (4,  "CC-1004", "Gas-TTF Desk",           "Trading",    "CWE",      True),
    (5,  "CC-1005", "Gas-NBP Desk",           "Trading",    "UK",       True),
    (6,  "CC-1006", "Oil-Brent Desk",         "Trading",    "Group",    True),
    (7,  "CC-1007", "Carbon-EUA Desk",        "Trading",    "Group",    True),
    (8,  "CC-1008", "Renewables Desk",        "Trading",    "Nordics",  True),
    (9,  "CC-1009", "LNG-Global Desk",        "Trading",    "Group",    True),
    (10, "CC-2001", "Power Generation Ops",   "Generation", "Nordics",  False),
    (11, "CC-2002", "LNG Terminal Ops",       "Generation", "UK",       False),
    (12, "CC-2003", "Renewables Assets",      "Generation", "Nordics",  False),
    (13, "CC-3001", "Executive Office",       "Corporate",  "Group",    False),
    (14, "CC-3002", "Finance & Accounting",   "Corporate",  "Group",    False),
    (15, "CC-3003", "Risk & Compliance",      "Corporate",  "Group",    False),
    (16, "CC-3004", "IT & Systems",           "Corporate",  "Group",    False),
    (17, "CC-3005", "Human Resources",        "Corporate",  "Group",    False),
    (18, "CC-3006", "Legal",                  "Corporate",  "Group",    False),
    (19, "CC-3007", "Operations & Settlement","Corporate",  "Group",    False),
]
dim_cost_center = spark.createDataFrame(
    cost_centers, "cost_center_key INT, cost_center_code STRING, cost_center_name STRING, "
                  "function STRING, region STRING, is_front_office BOOLEAN")
dim_cost_center.write.mode("overwrite").saveAsTable("dim_cost_center")
print(f"dim_cost_center: {dim_cost_center.count()} rows")

dim_cost_center: 19 rows


In [0]:
# (code, name, account_type, account_category, statement, normal_balance, is_pnl)
accounts_src = [
    ("1000", "Cash & Cash Equivalents",   "Asset",     "Current Assets",         "Balance Sheet",   "Debit",  False),
    ("1100", "Accounts Receivable",       "Asset",     "Current Assets",         "Balance Sheet",   "Debit",  False),
    ("1200", "Trading Receivables",       "Asset",     "Current Assets",         "Balance Sheet",   "Debit",  False),
    ("1300", "Fuel & Gas Inventory",      "Asset",     "Current Assets",         "Balance Sheet",   "Debit",  False),
    ("1500", "Property, Plant & Equipment","Asset",    "Non-Current Assets",     "Balance Sheet",   "Debit",  False),
    ("1600", "Accumulated Depreciation",  "Asset",     "Non-Current Assets",     "Balance Sheet",   "Credit", False),
    ("1700", "Goodwill & Intangibles",    "Asset",     "Non-Current Assets",     "Balance Sheet",   "Debit",  False),
    ("2000", "Accounts Payable",          "Liability", "Current Liabilities",    "Balance Sheet",   "Credit", False),
    ("2100", "Trading Payables",          "Liability", "Current Liabilities",    "Balance Sheet",   "Credit", False),
    ("2200", "Accrued Expenses",          "Liability", "Current Liabilities",    "Balance Sheet",   "Credit", False),
    ("2300", "Short-term Debt",           "Liability", "Current Liabilities",    "Balance Sheet",   "Credit", False),
    ("2500", "Long-term Debt",            "Liability", "Non-Current Liabilities","Balance Sheet",   "Credit", False),
    ("2600", "Tax Payable",               "Liability", "Current Liabilities",    "Balance Sheet",   "Credit", False),
    ("3000", "Share Capital",             "Equity",    "Equity",                 "Balance Sheet",   "Credit", False),
    ("3100", "Retained Earnings",         "Equity",    "Equity",                 "Balance Sheet",   "Credit", False),
    ("4000", "Power Sales Revenue",       "Revenue",   "Revenue",                "Income Statement","Credit", True),
    ("4100", "Gas Sales Revenue",         "Revenue",   "Revenue",                "Income Statement","Credit", True),
    ("4200", "LNG Sales Revenue",         "Revenue",   "Revenue",                "Income Statement","Credit", True),
    ("4300", "Realized Trading Gains",    "Revenue",   "Trading Revenue",        "Income Statement","Credit", True),
    ("4400", "Capacity & Ancillary Revenue","Revenue", "Revenue",               "Income Statement","Credit", True),
    ("5000", "Fuel Costs",                "Expense",   "Cost of Sales",          "Income Statement","Debit",  True),
    ("5100", "Purchased Power",           "Expense",   "Cost of Sales",          "Income Statement","Debit",  True),
    ("5200", "Transmission & Grid Fees",  "Expense",   "Cost of Sales",          "Income Statement","Debit",  True),
    ("5300", "Carbon & Emission Costs",   "Expense",   "Cost of Sales",          "Income Statement","Debit",  True),
    ("5400", "Realized Trading Losses",   "Expense",   "Cost of Sales",          "Income Statement","Debit",  True),
    ("6000", "Salaries & Wages",          "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("6100", "Employee Benefits",         "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("6200", "Depreciation & Amortization","Expense",  "Operating Expenses",     "Income Statement","Debit",  True),
    ("6300", "Maintenance & Repairs",     "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("6400", "SG&A",                      "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("6500", "IT & Systems",              "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("6600", "Professional Fees",         "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("6700", "Marketing",                 "Expense",   "Operating Expenses",     "Income Statement","Debit",  True),
    ("8000", "Interest Expense",          "Expense",   "Finance Costs",          "Income Statement","Debit",  True),
    ("8100", "Interest Income",           "Revenue",   "Other Income",           "Income Statement","Credit", True),
    ("8200", "FX Losses",                 "Expense",   "Finance Costs",          "Income Statement","Debit",  True),
    ("9000", "Income Tax Expense",        "Expense",   "Tax",                    "Income Statement","Debit",  True),
]
accounts = [(i + 1, *row) for i, row in enumerate(accounts_src)]
dim_account = spark.createDataFrame(
    accounts, "account_key INT, account_code STRING, account_name STRING, account_type STRING, "
              "account_category STRING, statement STRING, normal_balance STRING, is_pnl BOOLEAN")
dim_account.write.mode("overwrite").saveAsTable("dim_account")
acct_key = {code: key for (key, code, *_rest) in accounts}
print(f"dim_account: {dim_account.count()} rows")

dim_account: 37 rows


## Facts

In [0]:
import datetime

# Posting templates: (name, journal_type, source_system, debit_code, credit_code, function, base_eur, weight)
TEMPLATES = [
    ("Power sales invoice",       "Revenue",     "TradingSettle", "1100", "4000", "Trading",    850000, 10),
    ("Gas sales invoice",         "Revenue",     "TradingSettle", "1100", "4100", "Trading",    620000,  8),
    ("LNG cargo sale",            "Revenue",     "TradingSettle", "1100", "4200", "Generation",1400000,  4),
    ("Capacity/ancillary income", "Revenue",     "TradingSettle", "1100", "4400", "Generation", 180000,  4),
    ("Realized trading gain",     "Revenue",     "RiskSystem",    "1200", "4300", "Trading",    300000,  7),
    ("Fuel purchase",             "COGS",        "Procurement",   "5000", "2000", "Generation", 700000,  8),
    ("Purchased power",           "COGS",        "TradingSettle", "5100", "2000", "Trading",    500000,  7),
    ("Grid & transmission fees",  "COGS",        "Procurement",   "5200", "2000", "Generation",  90000,  4),
    ("Carbon/emission cost",      "COGS",        "Procurement",   "5300", "2000", "Generation", 160000,  4),
    ("Realized trading loss",     "COGS",        "RiskSystem",    "5400", "2100", "Trading",    220000,  5),
    ("Payroll — salaries",        "Payroll",     "HRPayroll",     "6000", "1000", "All",        240000,  6),
    ("Employee benefits accrual", "Payroll",     "HRPayroll",     "6100", "2200", "All",         60000,  4),
    ("Monthly depreciation",      "Depreciation","FixedAssets",   "6200", "1600", "Generation", 350000,  3),
    ("Maintenance & repairs",     "OpEx",        "AP",            "6300", "2000", "Generation", 120000,  4),
    ("SG&A administration",       "OpEx",        "AP",            "6400", "2000", "Corporate",   95000,  5),
    ("IT & systems",              "OpEx",        "AP",            "6500", "2000", "Corporate",   80000,  3),
    ("Professional fees",         "OpEx",        "AP",            "6600", "2000", "Corporate",   70000,  3),
    ("Marketing spend",           "OpEx",        "AP",            "6700", "2000", "Corporate",   40000,  2),
    ("Interest expense",          "Finance",     "Treasury",      "8000", "1000", "Corporate",  130000,  3),
    ("Interest income",           "Finance",     "Treasury",      "1000", "8100", "Corporate",   25000,  2),
    ("FX revaluation loss",       "Finance",     "Treasury",      "8200", "1000", "Corporate",   30000,  2),
    ("Income tax accrual",        "Tax",         "TaxEngine",     "9000", "2600", "Corporate",  210000,  2),
    ("Capex — PP&E addition",     "Capex",       "FixedAssets",   "1500", "1000", "Generation", 900000,  2),
    ("AP settlement",             "Payment",     "AP",            "2000", "1000", "Corporate",  400000,  6),
    ("AR collection",             "Receipt",     "AR",            "1000", "1100", "Corporate",  500000,  6),
]

# Template + weighted pool as small DataFrames
templ_rows = [
    (i, name, jtype, src, acct_key[dr], acct_key[cr], func, float(base))
    for i, (name, jtype, src, dr, cr, func, base, wt) in enumerate(TEMPLATES)
]
templates_df = spark.createDataFrame(
    templ_rows, "template_id INT, journal_name STRING, journal_type STRING, source_system STRING, "
                "dr_account_key INT, cr_account_key INT, function STRING, base_eur DOUBLE")

pool_rows, p = [], 0
for i, (*_, wt) in enumerate(TEMPLATES):
    for _ in range(wt):
        pool_rows.append((p, i)); p += 1
POOL = len(pool_rows)
pool_df = spark.createDataFrame(pool_rows, "pool_id INT, template_id INT")

# Cost-center pools by function (element_at picks one at random)
cc_map_rows = [
    ("Trading",    [k for (k, c, *_r) in cost_centers if c.startswith("CC-1")]),
    ("Generation", [k for (k, c, *_r) in cost_centers if c.startswith("CC-2")]),
    ("Corporate",  [k for (k, c, *_r) in cost_centers if c.startswith("CC-3")]),
    ("All",        [k for (k, *_r) in cost_centers]),
]
cc_map_df = spark.createDataFrame(cc_map_rows, "function STRING, cc_list ARRAY<INT>")

NUM_DAYS   = (datetime.date.fromisoformat(DATE_END) - datetime.date.fromisoformat(DATE_START)).days + 1
NUM_ENTITY = len(entities)

headers = (
    spark.range(0, NUM_JOURNALS).withColumnRenamed("id", "seq")
    .withColumn("pool_id", (F.rand(11) * POOL).cast("int"))
    .join(F.broadcast(pool_df), "pool_id")
    .join(F.broadcast(templates_df), "template_id")
    # entity + its currency / fx
    .withColumn("entity_key", (F.rand(12) * NUM_ENTITY).cast("int") + 1)
    .join(F.broadcast(dim_entity.select("entity_key", "currency_key", "fx_rate_to_eur")), "entity_key")
    # posting date within the demo window (bias to business days)
    .withColumn("day_off", (F.rand(13) * NUM_DAYS).cast("int"))
    .withColumn("posting_date", F.expr(f"date_add(to_date('{DATE_START}'), day_off)"))
    .withColumn("dow", F.dayofweek("posting_date"))
    # shift most weekend postings to the following Monday
    .withColumn("posting_date",
        F.when(F.col("dow").isin(1, 7) & (F.rand(16) < 0.8),
               F.date_add("posting_date", F.when(F.col("dow") == 7, 2).otherwise(1)))
         .otherwise(F.col("posting_date")))
    # Never let the weekend shift push a posting past DATE_END (keeps every date_key
    # inside dim_date and honours fk_journal_date).
    .withColumn("posting_date", F.least(F.col("posting_date"), F.to_date(F.lit(DATE_END))))
    .withColumn("date_key", F.date_format("posting_date", "yyyyMMdd").cast("int"))
    .withColumn("period", F.date_format("posting_date", "yyyy-MM"))
    # seasonality (energy: winter heavy) + gentle growth trend
    .withColumn("mth", F.month("posting_date"))
    .withColumn("months_since", (F.year("posting_date") - 2024) * 12 + F.month("posting_date") - 1)
    .withColumn("season", F.when(F.col("mth").isin(10, 11, 12, 1, 2, 3), F.lit(1.15)).otherwise(F.lit(0.90)))
    .withColumn("trend", F.lit(1.0) + F.lit(0.03) * F.col("months_since") / F.lit(12.0))
    .withColumn("mult", F.exp(F.randn(14) * F.lit(0.40)))
    .withColumn("amount_rpt", F.round(F.col("base_eur") * F.col("mult") * F.col("season") * F.col("trend"), 2))
    .withColumn("amount_lcl", F.round(F.col("amount_rpt") / F.col("fx_rate_to_eur"), 2))
    # cost center consistent with the template's function
    .join(F.broadcast(cc_map_df), "function")
    .withColumn("cost_center_key", F.element_at("cc_list", (F.rand(15) * F.size("cc_list")).cast("int") + 1))
    .withColumn("journal_id",
        F.concat(F.lit("JRN-"), F.date_format("posting_date", "yyyy"), F.lit("-"),
                 F.lpad((F.col("seq") + 1).cast("string"), 7, "0")))
)

# Explode each header into a debit and a credit line (signed: debit +, credit −)
entry = F.explode(F.array(
    F.struct(F.col("dr_account_key").alias("account_key"), F.lit(1).alias("line_no"),
             F.lit("Debit").alias("entry_type"),
             F.col("amount_lcl").alias("s_lcl"), F.col("amount_rpt").alias("s_rpt")),
    F.struct(F.col("cr_account_key").alias("account_key"), F.lit(2).alias("line_no"),
             F.lit("Credit").alias("entry_type"),
             (-F.col("amount_lcl")).alias("s_lcl"), (-F.col("amount_rpt")).alias("s_rpt")),
))

fact_gl_journal = (
    headers.withColumn("e", entry)
    .select(
        "journal_id",
        F.col("e.line_no").alias("line_no"),
        "date_key", "posting_date", "period",
        F.col("e.account_key").alias("account_key"),
        "cost_center_key", "entity_key", "currency_key",
        F.lit(ACTUAL_KEY).alias("scenario_key"),
        "journal_type", "source_system",
        F.col("journal_name").alias("description"),
        F.col("e.entry_type").alias("entry_type"),
        F.when(F.col("e.entry_type") == "Debit",  F.col("e.s_lcl")).otherwise(F.lit(0.0)).alias("debit_amount_lcl"),
        F.when(F.col("e.entry_type") == "Credit", -F.col("e.s_lcl")).otherwise(F.lit(0.0)).alias("credit_amount_lcl"),
        F.col("e.s_lcl").alias("signed_amount_lcl"),
        F.when(F.col("e.entry_type") == "Debit",  F.col("e.s_rpt")).otherwise(F.lit(0.0)).alias("debit_amount_rpt"),
        F.when(F.col("e.entry_type") == "Credit", -F.col("e.s_rpt")).otherwise(F.lit(0.0)).alias("credit_amount_rpt"),
        F.col("e.s_rpt").alias("signed_amount_rpt"),
    )
)
fact_gl_journal.write.mode("overwrite").saveAsTable("fact_gl_journal")
print(f"fact_gl_journal: {spark.table('fact_gl_journal').count():,} lines")

fact_gl_journal: 240,000 lines


In [0]:
jrnl = spark.table("fact_gl_journal")

monthly = (
    jrnl.groupBy("account_key", "entity_key", "cost_center_key", "period")
        .agg(F.sum("signed_amount_rpt").alias("period_movement_rpt"),
             F.sum("debit_amount_rpt").alias("period_debit_rpt"),
             F.sum("credit_amount_rpt").alias("period_credit_rpt"))
)

w_cum = (Window.partitionBy("account_key", "entity_key", "cost_center_key")
         .orderBy("period").rowsBetween(Window.unboundedPreceding, 0))

fact_gl_balance = (
    monthly
    .withColumn("closing_balance_rpt", F.round(F.sum("period_movement_rpt").over(w_cum), 2))
    .withColumn("opening_balance_rpt", F.round(F.col("closing_balance_rpt") - F.col("period_movement_rpt"), 2))
    .withColumn("period_end_key",
        F.date_format(F.last_day(F.to_date(F.concat(F.col("period"), F.lit("-01")))), "yyyyMMdd").cast("int"))
    .withColumn("scenario_key", F.lit(ACTUAL_KEY))
    .select("period", "period_end_key", "account_key", "entity_key", "cost_center_key", "scenario_key",
            "opening_balance_rpt", "period_debit_rpt", "period_credit_rpt",
            "period_movement_rpt", "closing_balance_rpt")
)
fact_gl_balance.write.mode("overwrite").saveAsTable("fact_gl_balance")
print(f"fact_gl_balance: {spark.table('fact_gl_balance').count():,} rows")

fact_gl_balance: 45,819 rows


In [0]:
# Budget target methodology (per account / entity / cost center / month):
#   • If a PRIOR-YEAR same-month actual exists, grow it:
#       income (Revenue) accounts → +3% to +10%   (random within the band)
#       cost   (Expense) accounts → −5%            (fixed)
#   • If there is NO prior year, base it on the CURRENT month's actual:
#       income accounts → +0% to +3%
#       cost   accounts → −0% to −2%
# Forecast stays a near-term re-estimate: current actual ± a small variance.
dim_acct = spark.table("dim_account").select("account_key", "account_type", "is_pnl")

# Monthly P&L actuals (signed movement) by account / entity / cost center / period
pnl_monthly = (monthly.join(dim_acct, "account_key")
                      .where(F.col("is_pnl"))
                      .select("account_key", "entity_key", "cost_center_key", "period",
                              "account_type", F.col("period_movement_rpt").alias("cur_amt")))

# Prior-year same month: shift each actual's period forward 12 months so it lines up
# with the target period it should serve as the budget base for.
prior = (pnl_monthly
    .withColumn("period",
        F.date_format(F.add_months(F.to_date(F.concat(F.col("period"), F.lit("-01"))), 12), "yyyy-MM"))
    .select("account_key", "entity_key", "cost_center_key", "period",
            F.col("cur_amt").alias("prior_amt")))

base = pnl_monthly.join(prior, ["account_key", "entity_key", "cost_center_key", "period"], "left")

is_income = F.col("account_type") == "Revenue"
has_prior = F.col("prior_amt").isNotNull()

budget_amt = (
    F.when(has_prior &  is_income, F.col("prior_amt") * (F.lit(1.0) + F.lit(0.03) + F.rand(31) * F.lit(0.07)))
     .when(has_prior & ~is_income, F.col("prior_amt") * F.lit(0.95))
     .when(is_income,              F.col("cur_amt")  * (F.lit(1.0) + F.rand(32) * F.lit(0.03)))
     .otherwise(                   F.col("cur_amt")  * (F.lit(1.0) - F.rand(33) * F.lit(0.02)))
)

budget = (base
    .withColumn("scenario_key", F.lit(BUDGET_KEY))
    .withColumn("plan_amount_rpt", F.round(budget_amt, 2))
    .select("account_key", "entity_key", "cost_center_key", "period", "scenario_key", "plan_amount_rpt"))

forecast = (pnl_monthly
    .withColumn("scenario_key", F.lit(FORECAST_KEY))
    .withColumn("plan_amount_rpt", F.round(F.col("cur_amt") * (F.lit(1.0) + F.randn(34) * F.lit(0.04)), 2))
    .select("account_key", "entity_key", "cost_center_key", "period", "scenario_key", "plan_amount_rpt"))

fact_financial_plan = (
    budget.unionByName(forecast)
    .withColumn("period_end_key",
        F.date_format(F.last_day(F.to_date(F.concat(F.col("period"), F.lit("-01")))), "yyyyMMdd").cast("int"))
    .select("period", "period_end_key", "account_key", "entity_key", "cost_center_key",
            "scenario_key", "plan_amount_rpt")
)
fact_financial_plan.write.mode("overwrite").saveAsTable("fact_financial_plan")
print(f"fact_financial_plan: {spark.table('fact_financial_plan').count():,} rows")

fact_financial_plan: 54,402 rows


## Validation & sample CPM queries

In [0]:
%sql
SELECT ROUND(SUM(signed_amount_rpt), 2) AS trial_balance_eur,
       COUNT(*)                          AS journal_lines,
       COUNT(DISTINCT journal_id)        AS journals
FROM fact_gl_journal;

trial_balance_eur,journal_lines,journals
0.0,240000,120000


In [0]:
%sql
SELECT d.fiscal_year,
       d.fiscal_quarter,
       ROUND(SUM(CASE WHEN a.account_type = 'Revenue' THEN -j.signed_amount_rpt ELSE 0 END), 0) AS revenue,
       ROUND(SUM(CASE WHEN a.account_type = 'Expense' THEN  j.signed_amount_rpt ELSE 0 END), 0) AS expenses,
       ROUND(-SUM(CASE WHEN a.is_pnl THEN j.signed_amount_rpt ELSE 0 END), 0)                   AS net_income
FROM fact_gl_journal j
JOIN dim_account a ON j.account_key = a.account_key
JOIN dim_date    d ON j.date_key    = d.date_key
GROUP BY d.fiscal_year, d.fiscal_quarter
ORDER BY d.fiscal_year, d.fiscal_quarter;

fiscal_year,fiscal_quarter,revenue,expenses,net_income
2024,Q2,2.628849105E9,1.963606475E9,6.6524263E8
2024,Q3,2.083503703E9,1.558410872E9,5.25092831E8
2024,Q4,2.186346402E9,1.685558381E9,5.00788021E8
2025,Q1,2.920527741E9,2.021136243E9,8.99391498E8
2025,Q2,2.729379798E9,2.064182134E9,6.65197663E8
2025,Q3,2.116448989E9,1.659595911E9,4.56853078E8
2025,Q4,2.281449912E9,1.683192756E9,5.98257156E8
2026,Q1,2.818547263E9,2.136859082E9,6.81688181E8
2026,Q2,2.902527939E9,2.154272455E9,7.48255485E8
2026,Q3,2.228217709E9,1.640094496E9,5.88123213E8


In [0]:
%sql
WITH actual AS (
  SELECT period, ROUND(-SUM(signed_amount_rpt), 0) AS actual_rev
  FROM fact_gl_journal j JOIN dim_account a ON j.account_key = a.account_key
  WHERE a.account_type = 'Revenue'
  GROUP BY period
),
plan AS (
  SELECT period,
         ROUND(-SUM(CASE WHEN scenario_key = 2 THEN plan_amount_rpt END), 0) AS budget_rev,
         ROUND(-SUM(CASE WHEN scenario_key = 3 THEN plan_amount_rpt END), 0) AS forecast_rev
  FROM fact_financial_plan p JOIN dim_account a ON p.account_key = a.account_key
  WHERE a.account_type = 'Revenue'
  GROUP BY period
)
SELECT a.period, a.actual_rev, pl.budget_rev, pl.forecast_rev,
       ROUND(a.actual_rev - pl.budget_rev, 0) AS variance_vs_budget
FROM actual a JOIN plan pl ON a.period = pl.period
ORDER BY a.period;

period,actual_rev,budget_rev,forecast_rev,variance_vs_budget
2024-01,8.89897045E8,9.03261638E8,8.88479456E8,-1.3364593E7
2024-02,8.25246762E8,8.3791859E8,8.29188084E8,-1.2671828E7
2024-03,9.13705298E8,9.27691143E8,9.10238929E8,-1.3985845E7
2024-04,7.47172973E8,7.58847955E8,7.45659295E8,-1.1674982E7
2024-05,7.15083783E8,7.25759238E8,7.1678882E8,-1.0675455E7
2024-06,6.21246947E8,6.31123724E8,6.21901714E8,-9876777.0
2024-07,7.04706998E8,7.15212249E8,7.0555838E8,-1.0505251E7
2024-08,7.1649411E8,7.26385759E8,7.12830766E8,-9891649.0
2024-09,7.65145293E8,7.77086789E8,7.63122475E8,-1.1941496E7
2024-10,9.35828576E8,9.5111452E8,9.34928605E8,-1.5285944E7


In [0]:
for t in ["dim_date", "dim_account", "dim_cost_center", "dim_entity", "dim_scenario",
          "dim_currency", "fact_gl_journal", "fact_gl_balance", "fact_financial_plan"]:
    print(f"{t:22s} {spark.table(t).count():>12,}")

dim_date                        974
dim_account                      37
dim_cost_center                  19
dim_entity                        6
dim_scenario                      3
dim_currency                      6
fact_gl_journal             240,000
fact_gl_balance              45,819
fact_financial_plan          54,402


## Part 5.2 — GL Primary & Foreign Keys

# Kong GL — Primary Key & Foreign Key Constraints

Adds Unity Catalog primary and foreign keys to the Kong Corporation general-ledger
star schema. Constraints are informational (NOT ENFORCED) but power BI tools, Genie,
and query optimization. Configurable via the **`catalog`** and **`gl_schema`** widgets
(defaults: `classic_stable_mags1.kong_gl`). All statements are idempotent (drop-if-exists
before add), so the notebook is safe to re-run.

## 1. Set primary-key columns NOT NULL

In [0]:
%sql
-- Dimension surrogate keys
ALTER TABLE ${catalog}.${gl_schema}.dim_date        ALTER COLUMN date_key        SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.dim_account     ALTER COLUMN account_key     SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.dim_cost_center ALTER COLUMN cost_center_key SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.dim_entity      ALTER COLUMN entity_key      SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.dim_currency    ALTER COLUMN currency_key    SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.dim_scenario    ALTER COLUMN scenario_key    SET NOT NULL;

-- Fact composite-key columns
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal     ALTER COLUMN journal_id      SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal     ALTER COLUMN line_no         SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     ALTER COLUMN account_key     SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     ALTER COLUMN entity_key      SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     ALTER COLUMN cost_center_key SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     ALTER COLUMN period_end_key  SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     ALTER COLUMN scenario_key    SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ALTER COLUMN account_key     SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ALTER COLUMN entity_key      SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ALTER COLUMN cost_center_key SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ALTER COLUMN period_end_key  SET NOT NULL;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ALTER COLUMN scenario_key    SET NOT NULL;

## 2. Primary keys on dimensions

In [0]:
%sql
ALTER TABLE ${catalog}.${gl_schema}.dim_date        DROP CONSTRAINT IF EXISTS pk_dim_date;
ALTER TABLE ${catalog}.${gl_schema}.dim_date        ADD  CONSTRAINT pk_dim_date        PRIMARY KEY (date_key);
ALTER TABLE ${catalog}.${gl_schema}.dim_account     DROP CONSTRAINT IF EXISTS pk_dim_account;
ALTER TABLE ${catalog}.${gl_schema}.dim_account     ADD  CONSTRAINT pk_dim_account     PRIMARY KEY (account_key);
ALTER TABLE ${catalog}.${gl_schema}.dim_cost_center DROP CONSTRAINT IF EXISTS pk_dim_cost_center;
ALTER TABLE ${catalog}.${gl_schema}.dim_cost_center ADD  CONSTRAINT pk_dim_cost_center PRIMARY KEY (cost_center_key);
ALTER TABLE ${catalog}.${gl_schema}.dim_entity      DROP CONSTRAINT IF EXISTS pk_dim_entity;
ALTER TABLE ${catalog}.${gl_schema}.dim_entity      ADD  CONSTRAINT pk_dim_entity      PRIMARY KEY (entity_key);
ALTER TABLE ${catalog}.${gl_schema}.dim_currency    DROP CONSTRAINT IF EXISTS pk_dim_currency;
ALTER TABLE ${catalog}.${gl_schema}.dim_currency    ADD  CONSTRAINT pk_dim_currency    PRIMARY KEY (currency_key);
ALTER TABLE ${catalog}.${gl_schema}.dim_scenario    DROP CONSTRAINT IF EXISTS pk_dim_scenario;
ALTER TABLE ${catalog}.${gl_schema}.dim_scenario    ADD  CONSTRAINT pk_dim_scenario    PRIMARY KEY (scenario_key);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6044743966222369>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'ALTER TABLE classic_stable_magh.kong_gl.dim_date        DROP CONSTRAINT IF EXISTS pk_dim_date;\nALTER TABLE classic_stable_magh.kong_gl.dim_date        ADD  CONSTRAINT pk_dim_date        PRIMARY KEY (date_key);\nALTER TABLE classic_stable_magh.kong_gl.dim_account     DROP CONSTRAINT IF EXISTS pk_dim_account;\nALTER TABLE classic_stable_magh.kong_gl.dim_account     ADD  CONSTRAINT pk_dim_account     PRIMARY KEY (account_key);\nALTER TABLE classic_stable_magh.kong_gl.dim_cost_center DROP CONSTRAINT IF EXISTS pk_dim_cost_center;\nALTER TABLE classic_stable_magh.kong_gl.dim_cost_center ADD  CONSTRAINT pk_dim_cost_center PRIMARY KEY (cost_center_key);\nALTER TABLE classic_stable_magh.kong_gl.dim_entity      DROP CONSTRAINT IF EXISTS pk_dim_entity;\nALTER

## 3. Primary keys on facts

In [0]:
%sql
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal     DROP CONSTRAINT IF EXISTS pk_fact_gl_journal;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal     ADD  CONSTRAINT pk_fact_gl_journal PRIMARY KEY (journal_id, line_no);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     DROP CONSTRAINT IF EXISTS pk_fact_gl_balance;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance     ADD  CONSTRAINT pk_fact_gl_balance PRIMARY KEY (account_key, entity_key, cost_center_key, period_end_key, scenario_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan DROP CONSTRAINT IF EXISTS pk_fact_financial_plan;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ADD  CONSTRAINT pk_fact_financial_plan PRIMARY KEY (account_key, entity_key, cost_center_key, period_end_key, scenario_key);

## 4. Dimension-to-dimension foreign key

In [0]:
%sql
-- Each entity has a functional currency
ALTER TABLE ${catalog}.${gl_schema}.dim_entity DROP CONSTRAINT IF EXISTS fk_entity_currency;
ALTER TABLE ${catalog}.${gl_schema}.dim_entity ADD  CONSTRAINT fk_entity_currency
  FOREIGN KEY (currency_key) REFERENCES ${catalog}.${gl_schema}.dim_currency(currency_key);

## 5. Foreign keys on fact_gl_journal

In [0]:
%sql
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal DROP CONSTRAINT IF EXISTS fk_journal_date;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal ADD  CONSTRAINT fk_journal_date        FOREIGN KEY (date_key)        REFERENCES ${catalog}.${gl_schema}.dim_date(date_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal DROP CONSTRAINT IF EXISTS fk_journal_account;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal ADD  CONSTRAINT fk_journal_account     FOREIGN KEY (account_key)     REFERENCES ${catalog}.${gl_schema}.dim_account(account_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal DROP CONSTRAINT IF EXISTS fk_journal_cost_center;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal ADD  CONSTRAINT fk_journal_cost_center FOREIGN KEY (cost_center_key) REFERENCES ${catalog}.${gl_schema}.dim_cost_center(cost_center_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal DROP CONSTRAINT IF EXISTS fk_journal_entity;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal ADD  CONSTRAINT fk_journal_entity      FOREIGN KEY (entity_key)      REFERENCES ${catalog}.${gl_schema}.dim_entity(entity_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal DROP CONSTRAINT IF EXISTS fk_journal_currency;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal ADD  CONSTRAINT fk_journal_currency    FOREIGN KEY (currency_key)    REFERENCES ${catalog}.${gl_schema}.dim_currency(currency_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal DROP CONSTRAINT IF EXISTS fk_journal_scenario;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_journal ADD  CONSTRAINT fk_journal_scenario    FOREIGN KEY (scenario_key)    REFERENCES ${catalog}.${gl_schema}.dim_scenario(scenario_key);

## 6. Foreign keys on fact_gl_balance and fact_financial_plan

In [0]:
%sql
-- fact_gl_balance
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance DROP CONSTRAINT IF EXISTS fk_balance_date;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance ADD  CONSTRAINT fk_balance_date        FOREIGN KEY (period_end_key)  REFERENCES ${catalog}.${gl_schema}.dim_date(date_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance DROP CONSTRAINT IF EXISTS fk_balance_account;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance ADD  CONSTRAINT fk_balance_account     FOREIGN KEY (account_key)     REFERENCES ${catalog}.${gl_schema}.dim_account(account_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance DROP CONSTRAINT IF EXISTS fk_balance_cost_center;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance ADD  CONSTRAINT fk_balance_cost_center FOREIGN KEY (cost_center_key) REFERENCES ${catalog}.${gl_schema}.dim_cost_center(cost_center_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance DROP CONSTRAINT IF EXISTS fk_balance_entity;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance ADD  CONSTRAINT fk_balance_entity      FOREIGN KEY (entity_key)      REFERENCES ${catalog}.${gl_schema}.dim_entity(entity_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance DROP CONSTRAINT IF EXISTS fk_balance_scenario;
ALTER TABLE ${catalog}.${gl_schema}.fact_gl_balance ADD  CONSTRAINT fk_balance_scenario    FOREIGN KEY (scenario_key)    REFERENCES ${catalog}.${gl_schema}.dim_scenario(scenario_key);

-- fact_financial_plan
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan DROP CONSTRAINT IF EXISTS fk_plan_date;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ADD  CONSTRAINT fk_plan_date        FOREIGN KEY (period_end_key)  REFERENCES ${catalog}.${gl_schema}.dim_date(date_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan DROP CONSTRAINT IF EXISTS fk_plan_account;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ADD  CONSTRAINT fk_plan_account     FOREIGN KEY (account_key)     REFERENCES ${catalog}.${gl_schema}.dim_account(account_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan DROP CONSTRAINT IF EXISTS fk_plan_cost_center;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ADD  CONSTRAINT fk_plan_cost_center FOREIGN KEY (cost_center_key) REFERENCES ${catalog}.${gl_schema}.dim_cost_center(cost_center_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan DROP CONSTRAINT IF EXISTS fk_plan_entity;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ADD  CONSTRAINT fk_plan_entity      FOREIGN KEY (entity_key)      REFERENCES ${catalog}.${gl_schema}.dim_entity(entity_key);
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan DROP CONSTRAINT IF EXISTS fk_plan_scenario;
ALTER TABLE ${catalog}.${gl_schema}.fact_financial_plan ADD  CONSTRAINT fk_plan_scenario    FOREIGN KEY (scenario_key)    REFERENCES ${catalog}.${gl_schema}.dim_scenario(scenario_key);

## 7. Verify constraints

In [0]:
%sql
SELECT constraint_name, constraint_type, table_name
FROM ${catalog}.information_schema.table_constraints
WHERE table_schema = '${gl_schema}'
ORDER BY table_name, constraint_type, constraint_name;

## Part 5.3 — GL Corporate Reporting Metric View

# Kong GL — Corporate Reporting Metric View

A governed **metric view** over the general-ledger journal fact for Corporate
Performance Management: P&L build-up (Revenue → Gross Profit → EBITDA → Operating
Income → Net Income), margins, and rolling / month-over-month trends, sliceable by
account, entity, cost center, function, scenario, and fiscal period.

All monetary measures are in the **EUR reporting currency** (`*_rpt`). Configurable via
the **`catalog`** and **`gl_schema`** widgets (defaults: `classic_stable_mags1.kong_gl`).

In [0]:
%sql
CREATE OR REPLACE VIEW ${catalog}.${gl_schema}.kong_gl_metrics
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Kong Corporation corporate reporting KPIs — GL journal fact joined to account, date, entity, cost center, and scenario dimensions. Revenue, margins, EBITDA, net income, and trend measures for CPM. All amounts in EUR reporting currency."

source: ${catalog}.${gl_schema}.fact_gl_journal

joins:
  - name: acct
    source: ${catalog}.${gl_schema}.dim_account
    on: source.account_key = acct.account_key
  - name: dt
    source: ${catalog}.${gl_schema}.dim_date
    on: source.date_key = dt.date_key
  - name: ent
    source: ${catalog}.${gl_schema}.dim_entity
    on: source.entity_key = ent.entity_key
  - name: cc
    source: ${catalog}.${gl_schema}.dim_cost_center
    on: source.cost_center_key = cc.cost_center_key
  - name: scn
    source: ${catalog}.${gl_schema}.dim_scenario
    on: source.scenario_key = scn.scenario_key

dimensions:
  # --- Time ---
  - name: Posting Date
    expr: dt.cal_date
    comment: "Calendar date the journal line was posted"
  - name: Month Start
    expr: DATE_TRUNC('MONTH', dt.cal_date)
    comment: "First day of the posting month (for trend windows)"
  - name: Period
    expr: dt.period
    comment: "Accounting period (yyyy-MM)"
  - name: Year
    expr: dt.year
    comment: "Calendar year"
  - name: Quarter
    expr: dt.quarter
    comment: "Calendar quarter (1-4)"
  - name: Fiscal Year
    expr: dt.fiscal_year
    comment: "Fiscal year (October start)"
  - name: Fiscal Quarter
    expr: dt.fiscal_quarter
    comment: "Fiscal quarter"
  # --- Account (chart of accounts) ---
  - name: Statement
    expr: acct.statement
    comment: "Financial statement (Balance Sheet / Income Statement)"
  - name: Account Type
    expr: acct.account_type
    comment: "Asset, Liability, Equity, Revenue, or Expense"
  - name: Account Category
    expr: acct.account_category
    comment: "Sub-classification of the account"
  - name: Account
    expr: acct.account_name
    comment: "GL account name"
  - name: Account Code
    expr: acct.account_code
    comment: "GL account code"
  # --- Organization ---
  - name: Entity
    expr: ent.entity_name
    comment: "Legal entity"
  - name: Entity Country
    expr: ent.country
    comment: "Country of the legal entity"
  - name: Region
    expr: ent.region
    comment: "Reporting region"
  - name: Cost Center
    expr: cc.cost_center_name
    comment: "Cost center"
  - name: Function
    expr: cc.function
    comment: "Cost center function (Trading / Generation / Corporate)"
  - name: Is Front Office
    expr: cc.is_front_office
    comment: "Whether the cost center is front office"
  - name: Scenario
    expr: scn.scenario_name
    comment: "Actual / Budget / Forecast"
  # --- Journal attributes ---
  - name: Journal Type
    expr: source.journal_type
    comment: "Type of journal (Revenue, COGS, Payroll, ...)"
  - name: Source System
    expr: source.source_system
    comment: "Originating source system"

measures:
  # --- Ledger integrity ---
  - name: Total Debits
    expr: SUM(source.debit_amount_rpt)
    comment: "Total debit postings (EUR)"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: Total Credits
    expr: SUM(source.credit_amount_rpt)
    comment: "Total credit postings (EUR)"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: Net Movement
    expr: SUM(source.signed_amount_rpt)
    comment: "Signed net movement (should be ~0 across the full ledger — trial balance)"
  - name: Journal Count
    expr: COUNT(DISTINCT source.journal_id)
    comment: "Number of distinct journal entries"
  # --- P&L build-up (all EUR, natural sign) ---
  - name: Revenue
    expr: SUM(CASE WHEN acct.account_type = 'Revenue' AND acct.account_category <> 'Other Income' THEN -source.signed_amount_rpt ELSE 0 END)
    comment: "Operating revenue (credits shown positive); excludes non-operating Other Income such as interest income"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: COGS
    expr: SUM(CASE WHEN acct.account_category = 'Cost of Sales' THEN source.signed_amount_rpt ELSE 0 END)
    comment: "Cost of sales (positive expense)"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: Gross Profit
    expr: SUM(CASE WHEN (acct.account_type = 'Revenue' AND acct.account_category <> 'Other Income') OR acct.account_category = 'Cost of Sales' THEN -source.signed_amount_rpt ELSE 0 END)
    comment: "Operating revenue minus cost of sales"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: Operating Expenses
    expr: SUM(CASE WHEN acct.account_category = 'Operating Expenses' THEN source.signed_amount_rpt ELSE 0 END)
    comment: "Operating expenses (SG&A, D&A, IT, ...)"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: Depreciation
    expr: SUM(CASE WHEN acct.account_code = '6200' THEN source.signed_amount_rpt ELSE 0 END)
    comment: "Depreciation & amortization expense"
  - name: Operating Income
    expr: SUM(CASE WHEN (acct.account_type = 'Revenue' AND acct.account_category <> 'Other Income') OR acct.account_category IN ('Cost of Sales', 'Operating Expenses') THEN -source.signed_amount_rpt ELSE 0 END)
    comment: "EBIT — operating revenue less cost of sales and operating expenses (excludes finance items)"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: EBITDA
    expr: MEASURE(`Operating Income`) + MEASURE(`Depreciation`)
    comment: "Operating income with depreciation & amortization added back"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  - name: Net Income
    expr: SUM(CASE WHEN acct.is_pnl THEN -source.signed_amount_rpt ELSE 0 END)
    comment: "Bottom-line profit — all P&L accounts"
    format:
      type: currency
      currency_code: EUR
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
  # --- Margins ---
  - name: Gross Margin %
    expr: MEASURE(`Gross Profit`) / NULLIF(MEASURE(`Revenue`), 0) * 100
    comment: "Gross profit as a percentage of revenue"
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
  - name: EBITDA Margin %
    expr: MEASURE(`EBITDA`) / NULLIF(MEASURE(`Revenue`), 0) * 100
    comment: "EBITDA as a percentage of revenue"
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
  - name: Net Margin %
    expr: MEASURE(`Net Income`) / NULLIF(MEASURE(`Revenue`), 0) * 100
    comment: "Net income as a percentage of revenue"
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
  # --- Trends ---
  - name: Monthly Revenue
    expr: MEASURE(`Revenue`)
    comment: "Revenue for the current month"
    window:
      - order: Month Start
        range: current
        semiadditive: last
  - name: Trailing 2M Revenue
    expr: MEASURE(`Revenue`)
    comment: "Revenue over the current + prior month — a trailing 1-month window is inclusive of the current period"
    window:
      - order: Month Start
        range: trailing 1 month
        semiadditive: last
  - name: Prev Month Revenue
    expr: MEASURE(`Trailing 2M Revenue`) - MEASURE(`Monthly Revenue`)
    comment: "Prior month's revenue only — trailing 2-month total minus the current month, so MoM compares against the prior month alone"
  - name: MoM Revenue Growth %
    expr: (MEASURE(`Monthly Revenue`) - MEASURE(`Prev Month Revenue`)) / NULLIF(MEASURE(`Prev Month Revenue`), 0) * 100
    comment: "Month-over-month revenue growth"
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
  - name: Rolling 12M Net Income
    expr: MEASURE(`Net Income`)
    comment: "Trailing 12-month net income"
    window:
      - order: Month Start
        range: trailing 12 month
        semiadditive: last
$$;

In [0]:
%sql
SELECT
  `Fiscal Year`,
  `Fiscal Quarter`,
  MEASURE(`Revenue`)          AS revenue,
  MEASURE(`Gross Profit`)     AS gross_profit,
  MEASURE(`EBITDA`)           AS ebitda,
  MEASURE(`Net Income`)       AS net_income,
  MEASURE(`Net Margin %`)     AS net_margin_pct
FROM ${catalog}.${gl_schema}.kong_gl_metrics
WHERE `Scenario` = 'Actual'
GROUP BY `Fiscal Year`, `Fiscal Quarter`
ORDER BY `Fiscal Year`, `Fiscal Quarter`;

In [0]:
%sql
SELECT
  `Entity`,
  `Function`,
  MEASURE(`Revenue`)        AS revenue,
  MEASURE(`Operating Income`) AS operating_income,
  MEASURE(`EBITDA Margin %`)  AS ebitda_margin_pct
FROM ${catalog}.${gl_schema}.kong_gl_metrics
WHERE `Scenario` = 'Actual'
GROUP BY `Entity`, `Function`
ORDER BY revenue DESC;

## Part 5.4 — GL AI-Generated Descriptions

Same engine as Part 4, re-pointed at the `kong_gl` schema — documents the GL dimensions, facts, and the `kong_gl_metrics` metric view.

In [0]:
# DBTITLE 1,Part 5 — AI descriptions config (GL schema)
# Re-point the AI engine at the GL schema; model / dry_run / skip_documented reuse Part 4's widgets.
SCHEMA          = GL_SCHEMA
MODEL           = dbutils.widgets.get("model").strip()
DRY_RUN         = dbutils.widgets.get("dry_run") == "true"
SKIP_DOCUMENTED = dbutils.widgets.get("skip_documented") == "true"
# Corporate-ledger domain so GL comments are framed for finance, not trading.
DOMAIN = ("the general ledger of Kong Corporation, an energy company — for corporate "
          "performance management and financial reporting (chart of accounts, journal "
          "entries, monthly balances, budgets/forecasts, cost centers, legal entities, "
          "and Actual/Budget/Forecast scenarios)")
print(f"AI-documenting GL schema: {CATALOG}.{SCHEMA}")

In [0]:
import json
from pyspark.sql.functions import expr

def sql_lit(text: str) -> str:
    """Collapse whitespace and escape single quotes for a SQL string literal."""
    return " ".join(str(text).split()).replace("'", "''")

def list_objects():
    """Return [(name, table_type, is_view, comment)] for every object in the schema."""
    rows = spark.sql(f"""
        SELECT table_name, table_type, comment
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}'
        ORDER BY table_type, table_name
    """).collect()
    out = []
    for r in rows:
        is_view = "VIEW" in (r["table_type"] or "").upper()
        out.append((r["table_name"], r["table_type"], is_view, r["comment"]))
    return out

def list_columns(table_name: str):
    """Ordered [(column_name, data_type)] for a table/view."""
    rows = spark.sql(f"""
        SELECT column_name, data_type
        FROM {CATALOG}.information_schema.columns
        WHERE table_schema = '{SCHEMA}' AND table_name = '{table_name}'
        ORDER BY ordinal_position
    """).collect()
    return [(r["column_name"], r["data_type"]) for r in rows]

def sample_rows(table_name: str, n: int = 3) -> str:
    """A compact string of up to n sample rows (base tables only)."""
    try:
        pdf = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.`{table_name}` LIMIT {n}").toPandas()
        return pdf.to_json(orient="records")[:2000]
    except Exception as e:
        return f"(no sample available: {e})"

def view_definition(table_name: str) -> str:
    try:
        rows = spark.sql(f"""
            SELECT view_definition
            FROM {CATALOG}.information_schema.views
            WHERE table_schema = '{SCHEMA}' AND table_name = '{table_name}'
        """).collect()
        return (rows[0]["view_definition"] or "")[:3000] if rows else ""
    except Exception:
        return ""

def build_prompt(name, is_view, columns):
    cols_desc = "\n".join(f"  - {c} ({t})" for c, t in columns)
    kind = "metric view" if is_view else "table"
    context = (f"VIEW DEFINITION:\n{view_definition(name)}"
               if is_view else f"SAMPLE ROWS (JSON):\n{sample_rows(name)}")
    col_keys = ", ".join(f'"{c}"' for c, _ in columns)
    return (
        f"You are a data catalog assistant documenting a star-schema data warehouse for {DOMAIN}.\n\n"
        f"Document the {kind} `{CATALOG}.{SCHEMA}.{name}`.\n\n"
        f"COLUMNS:\n{cols_desc}\n\n{context}\n\n"
        "Return ONLY a JSON object with this exact shape and nothing else:\n"
        '{"table_description": "<2 to 5 sentences describing what this object represents, '
        'its grain, and how it is used for analytics and reporting>", '
        f'"columns": {{{col_keys}: "<2 to 3 sentences describing this column, its meaning, '
        'units where relevant, and any keys/relationships>"}}}\n'
        "Rules: table_description must be 2-5 sentences; every column description must be "
        "2-3 sentences; be specific to the domain described above; do not invent columns; "
        "return valid JSON only."
    )

objects = list_objects()
print(f"Found {len(objects)} objects in {CATALOG}.{SCHEMA}:")
for name, ttype, is_view, comment in objects:
    flag = " (documented)" if comment else ""
    print(f"  - {name:24s} [{ttype}]{flag}")

In [0]:
targets = [(n, tt, iv) for (n, tt, iv, cmt) in objects if not (SKIP_DOCUMENTED and cmt)]

meta = {}          # name -> {"is_view", "columns"}
prompt_rows = []   # (name, prompt)
raw = {}
if not targets:
    # Skip only THIS section — never dbutils.notebook.exit(), which would halt the
    # entire merged full-solution notebook so later parts (e.g. the Kong GL model)
    # would never run.
    print("Nothing to document (all objects already have comments) — skipping this section.")
else:
    for name, ttype, is_view in targets:
        cols = list_columns(name)
        meta[name] = {"is_view": is_view, "columns": [c for c, _ in cols]}
        prompt_rows.append((name, build_prompt(name, is_view, cols)))

    prompts_df = spark.createDataFrame(prompt_rows, "name STRING, prompt STRING")
    result_df = prompts_df.withColumn(
        "out",
        expr(f"to_json(ai_query('{MODEL}', prompt, failOnError => false))")
    )
    # Materialize once and pull to the driver (object count is small)
    raw = {r["name"]: r["out"] for r in result_df.select("name", "out").collect()}
    print(f"Generated responses for {len(raw)} objects.")

In [0]:
import re

def _extract_json(text: str):
    """Try to extract a JSON object from text that may include markdown fences."""
    # Try to find a fenced JSON block first
    m = re.search(r'```(?:json)?\s*\n?(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        return m.group(1)
    # Otherwise find the first { ... } block
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def parse_response(out_json: str):
    """out_json is to_json() of the ai_query struct: {result, errorMessage}."""
    try:
        outer = json.loads(out_json)
    except Exception as e:
        return None, f"could not parse ai_query envelope: {e}"
    err = outer.get("errorMessage") or outer.get("error")
    if err:
        return None, err
    resp = outer.get("result") or outer.get("response")
    if not resp:
        return None, "empty response"
    # Try direct JSON parse first
    try:
        return json.loads(resp), None
    except Exception:
        pass
    # Try extracting JSON from markdown/text response
    extracted = _extract_json(resp)
    if extracted:
        try:
            return json.loads(extracted), None
        except Exception as e:
            return None, f"extracted JSON is invalid: {e}"
    return None, "model did not return valid JSON (no JSON object found in response)"

parsed = {}   # name -> {"table": str, "columns": {col: str}}
errors = {}
for name, out_json in raw.items():
    data, err = parse_response(out_json)
    if err:
        errors[name] = err
        continue
    parsed[name] = {
        "table": (data.get("table_description") or "").strip(),
        "columns": {k: (v or "").strip() for k, v in (data.get("columns") or {}).items()},
    }

if errors:
    print("⚠️  Errors:")
    for n, e in errors.items():
        print(f"   {n}: {e}")
print(f"Parsed descriptions for {len(parsed)} objects.")

In [0]:
preview = []
for name, d in parsed.items():
    preview.append((name, "TABLE", d["table"]))
    for col in meta[name]["columns"]:
        preview.append((name, col, d["columns"].get(col, "⚠️ (missing — will be skipped)")))

preview_df = spark.createDataFrame(preview, "object STRING, target STRING, proposed_description STRING")
print("Review the proposed descriptions below. Set dry_run=false and re-run to apply.")
display(preview_df)

In [0]:
def apply_comments():
    applied, skipped, failed = 0, 0, []
    for name, d in parsed.items():
        is_view = meta[name]["is_view"]
        fq = f"{CATALOG}.{SCHEMA}.`{name}`"
        obj_kw = "VIEW" if is_view else "TABLE"

        # Object-level comment
        if d["table"]:
            try:
                spark.sql(f"COMMENT ON {obj_kw} {fq} IS '{sql_lit(d['table'])}'")
                applied += 1
            except Exception as e:
                failed.append((name, "TABLE", str(e)[:200]))

        # Column-level comments
        alter_kw = "ALTER VIEW" if is_view else "ALTER TABLE"
        for col in meta[name]["columns"]:
            desc = d["columns"].get(col)
            if not desc:
                skipped += 1
                continue
            try:
                spark.sql(f"{alter_kw} {fq} ALTER COLUMN `{col}` COMMENT '{sql_lit(desc)}'")
                applied += 1
            except Exception as e:
                failed.append((name, col, str(e)[:200]))
    return applied, skipped, failed

if DRY_RUN:
    print("DRY RUN — no comments written. Set the dry_run widget to 'false' and re-run to apply.")
else:
    applied, skipped, failed = apply_comments()
    print(f"✅ Applied {applied} comments.  Skipped {skipped} (no description).")
    if failed:
        print(f"⚠️  {len(failed)} failed (metric-view columns may not accept ALTER COLUMN):")
        for name, tgt, err in failed:
            print(f"   {name}.{tgt}: {err}")

In [0]:
if not DRY_RUN:
    display(spark.sql(f"""
        SELECT table_name AS object, 'TABLE' AS target, comment AS description
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}' AND comment IS NOT NULL
        UNION ALL
        SELECT table_name, column_name, comment
        FROM {CATALOG}.information_schema.columns
        WHERE table_schema = '{SCHEMA}' AND comment IS NOT NULL
        ORDER BY object, target
    """))